# Notebook 6 – Practical Missing-Input Baselines and Robustness Benchmarking

## Objective

This notebook extends the robustness analysis developed in Notebooks 4–5 by evaluating simple and operationally realistic fallback strategies for unavailable lagged traffic observations.

The analysis is designed to address the practical question of whether simple missing-input handling strategies can recover forecasting performance without failure-aware model retraining.

The evaluated strategies will include:

1. R0 – Clean-only LightGBM without mitigation.
2. Previous-day lag substitution.
3. Previous-week lag substitution.
4. Seasonal-naive fallback when required lagged inputs are unavailable.
5. R1 – Failure-aware LightGBM.
6. R2 – Failure-aware LightGBM with missingness indicators, where relevant.

Performance will be evaluated under the same controlled failure-validation framework used in Notebook 5 using WAPE, MAE, RMSE, bias, prediction availability, and event-level robustness statistics.

No new failure scenarios will be introduced. The purpose is to benchmark simple practitioner-oriented mitigation strategies against the previously developed failure-aware approach.

> **Reproducibility note.** This public version uses project-relative paths. By default, the project root is `/content/traffic-forecasting-failure-robustness`. To use another location (including Google Drive), set the environment variable `TRAFFIC_FORECASTING_ROOT` before running the notebook. Raw source archives belong in `data/raw/`; generated artifacts are written under `outputs/`.


In [ ]:
# ============================================================
# REPRODUCIBILITY PATH SETUP
# ============================================================

from pathlib import Path
import os

# Change TRAFFIC_FORECASTING_ROOT only if the project is stored
# somewhere other than the default Colab location.
PROJECT_ROOT = Path(
    os.environ.get(
        "TRAFFIC_FORECASTING_ROOT",
        "/content/traffic-forecasting-failure-robustness"
    )
)

RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DATA_ROOT)
print("Outputs      :", OUTPUT_ROOT)

# ============================================================
# NOTEBOOK 6
# Practical Missing-Input Baselines and Robustness Benchmarking
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

NB2_ROOT = OUTPUT_ROOT / "Notebook_2_Output"
NB3_ROOT = OUTPUT_ROOT / "Notebook_3_Output"
NB4_ROOT = OUTPUT_ROOT / "Notebook_4_Output"
NB5_ROOT = OUTPUT_ROOT / "Notebook_5_Output"
NB6_ROOT = OUTPUT_ROOT / "Notebook_6_Output"

NB6_DATA = NB6_ROOT / "Data"
NB6_RESULTS = NB6_ROOT / "Results"
NB6_TABLES = NB6_ROOT / "Tables"
NB6_FIGURES = NB6_ROOT / "Figures"

for folder in [
    NB6_ROOT,
    NB6_DATA,
    NB6_RESULTS,
    NB6_TABLES,
    NB6_FIGURES
]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NOTEBOOK 6 INITIALIZED")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nNotebook 6 output:")
print(NB6_ROOT)

print("\nFolder status:")
for p in [NB2_ROOT, NB3_ROOT, NB4_ROOT, NB5_ROOT, NB6_ROOT]:
    print(f"{'✓' if p.exists() else '✗'} {p}")

In [ ]:
# ============================================================
# CELL 3
# Inspect key Notebook 5 artifacts
# ============================================================

candidate_files = {
    "July controlled events":
        NB5_ROOT / "Data" / "Controlled_July_Failure_Validation_Events.csv",

    "July clean/injected rows":
        NB5_ROOT / "Data" / "Controlled_July_Clean_Injected_Validation_Rows.parquet",

    "R0 July predictions":
        NB5_ROOT / "Data" / "R0_Controlled_July_Predictions.parquet",

    "R1 July predictions":
        NB5_ROOT / "Data" / "R1_Controlled_July_Predictions.parquet",

    "July R0/R1/R2 summary":
        NB5_ROOT / "Tables" / "Table_R0_R1_R2_July_Model_Selection.csv",

    "Clean LightGBM model":
        NB3_ROOT / "Models" / "LightGBM_Clean_15min.txt",

    "Final R1 model":
        NB5_ROOT / "Models" / "FINAL_R1_FailureAware_LightGBM_JanJul_1200.txt",
}

print("=" * 80)
print("KEY FILE CHECK")
print("=" * 80)

for name, path in candidate_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:8s} | {name}")
    print(f"           {path}")

In [ ]:
# ============================================================
# CELL 4
# List Notebook 5 data artifacts
# ============================================================

data_dir = NB5_ROOT / "Data"

files = sorted([
    p for p in data_dir.rglob("*")
    if p.is_file()
])

print(f"Number of files found: {len(files)}\n")

for i, p in enumerate(files, start=1):
    size_mb = p.stat().st_size / (1024 ** 2)

    print(
        f"{i:02d}. "
        f"{p.name:<65} "
        f"{size_mb:10.2f} MB"
    )

In [ ]:
# ============================================================
# CELL 5
# Inspect schemas of relevant parquet/csv files
# ============================================================

def inspect_file(path):
    path = Path(path)

    print("\n" + "=" * 90)
    print(path.name)
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND")
        return

    if path.suffix.lower() == ".parquet":

        import pyarrow.parquet as pq

        pf = pq.ParquetFile(path)

        print(f"Rows       : {pf.metadata.num_rows:,}")
        print(f"Row groups : {pf.metadata.num_row_groups:,}")
        print(f"Columns    : {len(pf.schema.names)}")

        print("\nColumn names:")
        for c in pf.schema.names:
            print(" -", c)

    elif path.suffix.lower() == ".csv":

        df_head = pd.read_csv(path, nrows=5)

        print("Columns:")
        for c in df_head.columns:
            print(" -", c)

        print("\nPreview:")
        display(df_head)


files_to_inspect = [
    candidate_files["July controlled events"],
    candidate_files["July clean/injected rows"],
    candidate_files["R0 July predictions"],
    candidate_files["R1 July predictions"],
    candidate_files["July R0/R1/R2 summary"],
]

for f in files_to_inspect:
    inspect_file(f)

In [ ]:
# ============================================================
# CELL 6
# Audit July Controlled Validation Data
# ============================================================

controlled_path = (
    NB5_ROOT / "Data" /
    "Controlled_July_Clean_Injected_Validation_Rows.parquet"
)

controlled = pd.read_parquet(controlled_path)

controlled["Timestamp"] = pd.to_datetime(controlled["Timestamp"])

lag_cols = ["Lag_1", "Lag_2", "Lag_4", "Lag_8", "Lag_96"]
missing_cols = [f"Missing_{c}" for c in lag_cols]

print("=" * 75)
print("CONTROLLED JULY VALIDATION AUDIT")
print("=" * 75)

print(f"Rows              : {len(controlled):,}")
print(f"Events            : {controlled['July_Validation_Event_ID'].nunique():,}")
print(f"Detectors         : {controlled['Detector_Code'].nunique():,}")
print(f"Start timestamp   : {controlled['Timestamp'].min()}")
print(f"End timestamp     : {controlled['Timestamp'].max()}")

print("\nMissing-lag count distribution:")
print(
    controlled["Missing_Lag_Count"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nIndividual missing-lag frequencies:")
for c in missing_cols:
    n = int(controlled[c].sum())
    pct = 100 * n / len(controlled)
    print(f"{c:<18}: {n:>6,} ({pct:6.2f}%)")

print("\nMost common missing-lag patterns:")
display(
    controlled["Missing_Lag_Pattern"]
    .value_counts()
    .rename_axis("Pattern")
    .reset_index(name="Rows")
    .head(20)
)

print("\nInjected lag NaN check:")
for lag in lag_cols:
    c = f"{lag}_Injected"
    print(
        f"{c:<20}: "
        f"{controlled[c].isna().sum():>6,} NaNs"
    )

print("\nClean lag NaN check:")
for lag in lag_cols:
    c = f"{lag}_Clean"
    print(
        f"{c:<20}: "
        f"{controlled[c].isna().sum():>6,} NaNs"
    )

In [ ]:
# ============================================================
# CELL 7
# Locate Historical Traffic Data Candidates
# ============================================================

import pyarrow.parquet as pq

search_roots = [
    NB2_ROOT,
    NB3_ROOT,
    NB4_ROOT,
    NB5_ROOT
]

required_cols = {
    "Timestamp",
    "Detector_Code",
    "Target_Volume"
}

candidates = []

for root in search_roots:

    if not root.exists():
        continue

    for path in root.rglob("*.parquet"):

        try:
            pf = pq.ParquetFile(path)
            cols = set(pf.schema.names)

            if required_cols.issubset(cols):

                candidates.append({
                    "Notebook": root.name,
                    "File": path.name,
                    "Rows": pf.metadata.num_rows,
                    "Columns": len(cols),
                    "Size_MB": round(
                        path.stat().st_size / (1024 ** 2), 2
                    ),
                    "Path": str(path)
                })

        except Exception:
            pass

candidate_df = pd.DataFrame(candidates)

print("=" * 90)
print("HISTORICAL DATA CANDIDATES")
print("=" * 90)

if len(candidate_df) == 0:
    print("No matching parquet files found.")

else:

    candidate_df = candidate_df.sort_values(
        ["Rows", "Size_MB"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f"Number of candidates: {len(candidate_df)}\n")

    display(candidate_df)

In [ ]:
# ============================================================
# CELL 8
# Inspect June and July Historical Forecasting Data
# ============================================================

june_path = (
    NB3_ROOT / "Data" / "FORECASTING_2026-06.parquet"
)

july_path = (
    NB3_ROOT / "Data" / "FORECASTING_2026-07.parquet"
)

# If files are stored directly under Notebook_3_Output,
# automatically use that location instead.
if not june_path.exists():
    june_path = NB3_ROOT / "FORECASTING_2026-06.parquet"

if not july_path.exists():
    july_path = NB3_ROOT / "FORECASTING_2026-07.parquet"


def inspect_history_file(path, label):

    print("\n" + "=" * 85)
    print(label)
    print("=" * 85)

    if not path.exists():
        print("FILE NOT FOUND:")
        print(path)
        return

    pf = pq.ParquetFile(path)

    print(f"Path       : {path}")
    print(f"Rows       : {pf.metadata.num_rows:,}")
    print(f"Columns    : {len(pf.schema.names)}")

    print("\nColumns:")
    for c in pf.schema.names:
        print(" -", c)

    # Load only columns needed for basic validation
    check_cols = [
        "Timestamp",
        "Detector_Code",
        "Target_Volume"
    ]

    df_check = pd.read_parquet(
        path,
        columns=check_cols
    )

    df_check["Timestamp"] = pd.to_datetime(
        df_check["Timestamp"]
    )

    print("\nCoverage:")
    print(f"Start      : {df_check['Timestamp'].min()}")
    print(f"End        : {df_check['Timestamp'].max()}")
    print(
        f"Detectors  : "
        f"{df_check['Detector_Code'].nunique():,}"
    )

    duplicate_count = df_check.duplicated(
        ["Detector_Code", "Timestamp"]
    ).sum()

    print(
        f"Duplicate detector-timestamps : "
        f"{duplicate_count:,}"
    )

    print(
        f"Missing Target_Volume         : "
        f"{df_check['Target_Volume'].isna().sum():,}"
    )


inspect_history_file(
    june_path,
    "JUNE 2026 FORECASTING DATA"
)

inspect_history_file(
    july_path,
    "JULY 2026 FORECASTING DATA"
)

In [ ]:
# ============================================================
# CELL 8
# Resolve Exact June and July Forecasting File Paths
# ============================================================

target_files = [
    "FORECASTING_2026-06.parquet",
    "FORECASTING_2026-07.parquet"
]

resolved = candidate_df[
    candidate_df["File"].isin(target_files)
][["File", "Path", "Rows", "Columns", "Size_MB"]].copy()

print("=" * 90)
print("RESOLVED FORECASTING FILE PATHS")
print("=" * 90)

display(resolved)

for _, row in resolved.iterrows():
    print("\nFILE:")
    print(row["File"])
    print("PATH:")
    print(row["Path"])
    print("EXISTS:", Path(row["Path"]).exists())

In [ ]:
# ============================================================
# CELL 9
# Validate June and July Historical Traffic Data
# ============================================================

june_path = Path(
    resolved.loc[
        resolved["File"] == "FORECASTING_2026-06.parquet",
        "Path"
    ].iloc[0]
)

july_path = Path(
    resolved.loc[
        resolved["File"] == "FORECASTING_2026-07.parquet",
        "Path"
    ].iloc[0]
)

required_history_cols = [
    "Timestamp",
    "Detector_Code",
    "Target_Volume"
]

for label, path in [
    ("JUNE 2026", june_path),
    ("JULY 2026", july_path)
]:

    print("\n" + "=" * 85)
    print(label)
    print("=" * 85)

    pf = pq.ParquetFile(path)

    print(f"Rows    : {pf.metadata.num_rows:,}")
    print(f"Columns : {len(pf.schema.names)}")

    print("\nSchema:")
    for c in pf.schema.names:
        print(" -", c)

    hist_check = pd.read_parquet(
        path,
        columns=required_history_cols
    )

    hist_check["Timestamp"] = pd.to_datetime(
        hist_check["Timestamp"]
    )

    print("\nCoverage:")
    print(f"Start       : {hist_check['Timestamp'].min()}")
    print(f"End         : {hist_check['Timestamp'].max()}")
    print(f"Detectors   : {hist_check['Detector_Code'].nunique():,}")

    duplicates = hist_check.duplicated(
        ["Detector_Code", "Timestamp"]
    ).sum()

    missing_target = hist_check["Target_Volume"].isna().sum()

    print(f"Duplicates  : {duplicates:,}")
    print(f"Missing y   : {missing_target:,}")

    print("\nTarget volume summary:")
    print(
        hist_check["Target_Volume"]
        .describe()
        .round(3)
        .to_string()
    )

    del hist_check

In [ ]:
# ============================================================
# CELL 10
# Build Previous-Day and Previous-Week Substitution Values
# ============================================================

# Relevant detectors only
validation_detectors = set(
    controlled["Detector_Code"].unique()
)

# Load only required historical columns
history_cols = [
    "Timestamp",
    "Detector_Code",
    "Target_Volume"
]

june_hist = pd.read_parquet(
    june_path,
    columns=history_cols
)

july_hist = pd.read_parquet(
    july_path,
    columns=history_cols
)

# Combine June + July
history = pd.concat(
    [june_hist, july_hist],
    ignore_index=True
)

history["Timestamp"] = pd.to_datetime(history["Timestamp"])

# Keep only detectors used in controlled July validation
history = history[
    history["Detector_Code"].isin(validation_detectors)
].copy()

# Safety check
history = history.sort_values(
    ["Detector_Code", "Timestamp"]
).reset_index(drop=True)

duplicate_keys = history.duplicated(
    ["Detector_Code", "Timestamp"]
).sum()

print("=" * 80)
print("HISTORICAL LOOKUP DATA")
print("=" * 80)

print(f"Rows                 : {len(history):,}")
print(f"Detectors            : {history['Detector_Code'].nunique():,}")
print(f"Start                : {history['Timestamp'].min()}")
print(f"End                  : {history['Timestamp'].max()}")
print(f"Duplicate keys       : {duplicate_keys:,}")
print(f"Missing TargetVolume : {history['Target_Volume'].isna().sum():,}")

# ------------------------------------------------------------
# Build lookup dictionary using:
# (Detector_Code, Timestamp) -> Target_Volume
# ------------------------------------------------------------

history_lookup = history.set_index(
    ["Detector_Code", "Timestamp"]
)["Target_Volume"]

# Work on a copy of controlled validation rows
baseline_rows = controlled.copy()

lag_steps = {
    "Lag_1": 1,
    "Lag_2": 2,
    "Lag_4": 4,
    "Lag_8": 8,
    "Lag_96": 96
}

# ------------------------------------------------------------
# Generate substitute values
#
# Previous-day substitute:
# y(t - lag - 96)
#
# Previous-week substitute:
# y(t - lag - 672)
# ------------------------------------------------------------

for lag_name, lag_k in lag_steps.items():

    day_timestamp = (
        baseline_rows["Timestamp"]
        - pd.to_timedelta(
            (lag_k + 96) * 15,
            unit="m"
        )
    )

    week_timestamp = (
        baseline_rows["Timestamp"]
        - pd.to_timedelta(
            (lag_k + 672) * 15,
            unit="m"
        )
    )

    day_keys = pd.MultiIndex.from_arrays(
        [
            baseline_rows["Detector_Code"],
            day_timestamp
        ],
        names=["Detector_Code", "Timestamp"]
    )

    week_keys = pd.MultiIndex.from_arrays(
        [
            baseline_rows["Detector_Code"],
            week_timestamp
        ],
        names=["Detector_Code", "Timestamp"]
    )

    baseline_rows[
        f"{lag_name}_PrevDay"
    ] = history_lookup.reindex(day_keys).to_numpy()

    baseline_rows[
        f"{lag_name}_PrevWeek"
    ] = history_lookup.reindex(week_keys).to_numpy()


# ------------------------------------------------------------
# Availability audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SUBSTITUTION AVAILABILITY")
print("=" * 80)

for lag_name in lag_steps:

    day_col = f"{lag_name}_PrevDay"
    week_col = f"{lag_name}_PrevWeek"

    day_available = baseline_rows[day_col].notna().sum()
    week_available = baseline_rows[week_col].notna().sum()

    print(
        f"{lag_name:<7} | "
        f"Prev-Day: {day_available:>6,}/{len(baseline_rows):,} "
        f"({100*day_available/len(baseline_rows):6.2f}%) | "
        f"Prev-Week: {week_available:>6,}/{len(baseline_rows):,} "
        f"({100*week_available/len(baseline_rows):6.2f}%)"
    )

# ------------------------------------------------------------
# Check availability specifically where each lag is missing
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AVAILABILITY WHERE SUBSTITUTION IS ACTUALLY NEEDED")
print("=" * 80)

for lag_name in lag_steps:

    missing_flag = f"Missing_{lag_name}"

    needed = baseline_rows[missing_flag] == 1
    n_needed = needed.sum()

    day_ok = baseline_rows.loc[
        needed,
        f"{lag_name}_PrevDay"
    ].notna().sum()

    week_ok = baseline_rows.loc[
        needed,
        f"{lag_name}_PrevWeek"
    ].notna().sum()

    print(
        f"{lag_name:<7} | Needed: {n_needed:>5,} | "
        f"Prev-Day available: {day_ok:>5,} "
        f"({100*day_ok/n_needed:6.2f}%) | "
        f"Prev-Week available: {week_ok:>5,} "
        f"({100*week_ok/n_needed:6.2f}%)"
    )

In [ ]:
# ============================================================
# CELL 11
# Build Practical Imputed Lag Feature Sets
# ============================================================

baseline_eval = baseline_rows.copy()

lag_names = ["Lag_1", "Lag_2", "Lag_4", "Lag_8", "Lag_96"]

# ------------------------------------------------------------
# Create two practical input sets:
#
# B1 = Previous-Day substitution
# B2 = Previous-Week substitution
#
# Rule:
# - if injected lag is available -> keep it
# - if injected lag is missing -> replace it with historical
#   value from previous day/week
# ------------------------------------------------------------

for lag in lag_names:

    injected_col = f"{lag}_Injected"

    prev_day_col = f"{lag}_PrevDay"
    prev_week_col = f"{lag}_PrevWeek"

    b1_col = f"{lag}_B1_PrevDay"
    b2_col = f"{lag}_B2_PrevWeek"

    # B1: previous-day replacement
    baseline_eval[b1_col] = (
        baseline_eval[injected_col]
        .fillna(baseline_eval[prev_day_col])
    )

    # B2: previous-week replacement
    baseline_eval[b2_col] = (
        baseline_eval[injected_col]
        .fillna(baseline_eval[prev_week_col])
    )


# ------------------------------------------------------------
# Row-level completeness
# ------------------------------------------------------------

b1_cols = [f"{lag}_B1_PrevDay" for lag in lag_names]
b2_cols = [f"{lag}_B2_PrevWeek" for lag in lag_names]

baseline_eval["B1_All_Lags_Available"] = (
    baseline_eval[b1_cols].notna().all(axis=1)
)

baseline_eval["B2_All_Lags_Available"] = (
    baseline_eval[b2_cols].notna().all(axis=1)
)


print("=" * 80)
print("PRACTICAL BASELINE INPUT COMPLETENESS")
print("=" * 80)

# B1
b1_complete = baseline_eval["B1_All_Lags_Available"].sum()
b1_incomplete = len(baseline_eval) - b1_complete

print("\nB1 — Previous-Day Substitution")
print(f"Complete rows   : {b1_complete:,} / {len(baseline_eval):,}")
print(
    f"Availability    : "
    f"{100 * b1_complete / len(baseline_eval):.2f}%"
)
print(f"Incomplete rows : {b1_incomplete:,}")

# B2
b2_complete = baseline_eval["B2_All_Lags_Available"].sum()
b2_incomplete = len(baseline_eval) - b2_complete

print("\nB2 — Previous-Week Substitution")
print(f"Complete rows   : {b2_complete:,} / {len(baseline_eval):,}")
print(
    f"Availability    : "
    f"{100 * b2_complete / len(baseline_eval):.2f}%"
)
print(f"Incomplete rows : {b2_incomplete:,}")


# ------------------------------------------------------------
# Remaining missing values by lag
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REMAINING MISSING VALUES AFTER SUBSTITUTION")
print("=" * 80)

print("\nB1 — Previous-Day:")
for col in b1_cols:
    print(f"{col:<25}: {baseline_eval[col].isna().sum():>5,}")

print("\nB2 — Previous-Week:")
for col in b2_cols:
    print(f"{col:<25}: {baseline_eval[col].isna().sum():>5,}")


# ------------------------------------------------------------
# Event-level availability
# An event is fully available only if all its evaluation rows
# contain all required lag values after substitution.
# ------------------------------------------------------------

event_availability = (
    baseline_eval
    .groupby("July_Validation_Event_ID")
    .agg(
        B1_Event_Fully_Available=(
            "B1_All_Lags_Available", "all"
        ),
        B2_Event_Fully_Available=(
            "B2_All_Lags_Available", "all"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("EVENT-LEVEL AVAILABILITY")
print("=" * 80)

for col, label in [
    ("B1_Event_Fully_Available", "B1 Previous-Day"),
    ("B2_Event_Fully_Available", "B2 Previous-Week")
]:

    n = event_availability[col].sum()
    total = len(event_availability)

    print(
        f"{label:<20}: "
        f"{n:,}/{total:,} events "
        f"({100*n/total:.2f}%)"
    )

In [ ]:
# ============================================================
# CELL 12
# Load and Inspect Original R0 Clean-Only LightGBM Model
# ============================================================

import lightgbm as lgb

r0_model_path = (
    NB3_ROOT / "Models" / "LightGBM_Clean_15min.txt"
)

print("=" * 80)
print("LOAD ORIGINAL R0 LIGHTGBM MODEL")
print("=" * 80)

print(f"Model path : {r0_model_path}")
print(f"Exists     : {r0_model_path.exists()}")

# Load model
r0_model = lgb.Booster(
    model_file=str(r0_model_path)
)

# Inspect model features
r0_features = r0_model.feature_name()

print("\nNumber of model features:")
print(len(r0_features))

print("\nModel feature order:")
for i, feature in enumerate(r0_features, start=1):
    print(f"{i:02d}. {feature}")

print("\nBest iteration:")
print(r0_model.best_iteration)

print("\nNumber of trees:")
print(r0_model.num_trees())

In [ ]:
# ============================================================
# CELL 13
# Verify R0 Model Reproduces Stored Clean Predictions
# ============================================================

# Build clean feature matrix using the exact model feature order
X_clean_verify = pd.DataFrame({
    "Detector_Code": baseline_eval["Detector_Code"],
    "Interval_Index": baseline_eval["Interval_Index"],
    "DayOfWeek": baseline_eval["DayOfWeek"],
    "IsWeekend": baseline_eval["IsWeekend"],
    "Lag_1": baseline_eval["Lag_1_Clean"],
    "Lag_2": baseline_eval["Lag_2_Clean"],
    "Lag_4": baseline_eval["Lag_4_Clean"],
    "Lag_8": baseline_eval["Lag_8_Clean"],
    "Lag_96": baseline_eval["Lag_96_Clean"],
})

# Predict using original clean-only LightGBM model
r0_clean_reproduced = r0_model.predict(X_clean_verify)

# Load stored Notebook 5 R0 predictions
r0_saved_path = (
    NB5_ROOT / "Data" /
    "R0_Controlled_July_Predictions.parquet"
)

r0_saved = pd.read_parquet(r0_saved_path)

r0_saved["Timestamp"] = pd.to_datetime(
    r0_saved["Timestamp"]
)

# Join carefully using event, detector, timestamp
verification = baseline_eval[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ]
].copy()

verification["R0_Reproduced"] = r0_clean_reproduced

verification = verification.merge(
    r0_saved[
        [
            "July_Validation_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Clean_Prediction"
        ]
    ],
    on=[
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

# Compare predictions
verification["Absolute_Difference"] = (
    verification["R0_Reproduced"]
    - verification["R0_Clean_Prediction"]
).abs()

print("=" * 80)
print("R0 CLEAN PREDICTION REPRODUCIBILITY CHECK")
print("=" * 80)

print(f"Rows checked            : {len(verification):,}")
print(
    f"Stored predictions found: "
    f"{verification['R0_Clean_Prediction'].notna().sum():,}"
)

print(
    f"Maximum abs difference  : "
    f"{verification['Absolute_Difference'].max():.12f}"
)

print(
    f"Mean abs difference     : "
    f"{verification['Absolute_Difference'].mean():.12f}"
)

print(
    f"Rows difference > 1e-10 : "
    f"{(verification['Absolute_Difference'] > 1e-10).sum():,}"
)

print(
    f"Rows difference > 1e-8  : "
    f"{(verification['Absolute_Difference'] > 1e-8).sum():,}"
)

# Final verdict
max_diff = verification["Absolute_Difference"].max()

print("\n" + "=" * 80)

if max_diff < 1e-8:
    print("PASS — Original R0 predictions reproduced successfully.")
else:
    print("CHECK REQUIRED — Predictions do not exactly reproduce stored R0 results.")

print("=" * 80)

In [ ]:
# ============================================================
# CELL 14
# Diagnose R0 Reproduction Differences
# ============================================================

# Keep only rows with meaningful differences
diff_rows = verification[
    verification["Absolute_Difference"] > 1e-8
].copy()

# Add relevant clean features and target for inspection
diagnostic_cols = [
    "July_Validation_Event_ID",
    "Detector_Code",
    "Timestamp",
    "Target_Volume",
    "Interval_Index",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1_Clean",
    "Lag_2_Clean",
    "Lag_4_Clean",
    "Lag_8_Clean",
    "Lag_96_Clean",
]

diagnostic = baseline_eval[
    diagnostic_cols
].merge(
    diff_rows[
        [
            "July_Validation_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Reproduced",
            "R0_Clean_Prediction",
            "Absolute_Difference",
        ]
    ],
    on=[
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="inner",
    validate="one_to_one"
)

print("=" * 90)
print("R0 DIFFERENCE DIAGNOSTIC")
print("=" * 90)

print(f"Different rows: {len(diagnostic):,}")

print("\nReproduced prediction summary:")
print(
    diagnostic["R0_Reproduced"]
    .describe()
    .round(6)
    .to_string()
)

print("\nStored prediction summary:")
print(
    diagnostic["R0_Clean_Prediction"]
    .describe()
    .round(6)
    .to_string()
)

print("\nNegative reproduced predictions:")
print(
    (diagnostic["R0_Reproduced"] < 0).sum()
)

print("\nStored predictions equal to zero:")
print(
    np.isclose(
        diagnostic["R0_Clean_Prediction"],
        0.0,
        atol=1e-12
    ).sum()
)

# Test whether stored predictions correspond to clipping
diagnostic["R0_Reproduced_Clipped"] = np.maximum(
    diagnostic["R0_Reproduced"],
    0
)

diagnostic["Difference_After_Clipping"] = (
    diagnostic["R0_Reproduced_Clipped"]
    - diagnostic["R0_Clean_Prediction"]
).abs()

print("\n" + "=" * 90)
print("CLIPPING TEST")
print("=" * 90)

print(
    "Max difference after clipping : "
    f"{diagnostic['Difference_After_Clipping'].max():.12f}"
)

print(
    "Rows > 1e-8 after clipping    : "
    f"{(diagnostic['Difference_After_Clipping'] > 1e-8).sum():,}"
)

print("\nLargest differences:")
display(
    diagnostic.sort_values(
        "Absolute_Difference",
        ascending=False
    ).head(28)
)

In [ ]:
# ============================================================
# CELL 15
# Evaluate B1 — Previous-Day Lag Substitution
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# Build B1 feature matrix using EXACT R0 feature structure
# ------------------------------------------------------------

X_b1 = pd.DataFrame({
    "Detector_Code": baseline_eval["Detector_Code"],
    "Interval_Index": baseline_eval["Interval_Index"],
    "DayOfWeek": baseline_eval["DayOfWeek"],
    "IsWeekend": baseline_eval["IsWeekend"],
    "Lag_1": baseline_eval["Lag_1_B1_PrevDay"],
    "Lag_2": baseline_eval["Lag_2_B1_PrevDay"],
    "Lag_4": baseline_eval["Lag_4_B1_PrevDay"],
    "Lag_8": baseline_eval["Lag_8_B1_PrevDay"],
    "Lag_96": baseline_eval["Lag_96_B1_PrevDay"],
})

# Safety check
assert X_b1.isna().sum().sum() == 0, \
    "B1 contains missing feature values."

# ------------------------------------------------------------
# Predict with ORIGINAL R0 clean-only model
# No retraining
# ------------------------------------------------------------

b1_raw_pred = r0_model.predict(X_b1)

# Apply exact same post-processing used in Notebook 5
b1_pred = np.maximum(b1_raw_pred, 0)

actual = baseline_eval["Target_Volume"].to_numpy()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

b1_mae = mean_absolute_error(
    actual,
    b1_pred
)

b1_rmse = np.sqrt(
    mean_squared_error(
        actual,
        b1_pred
    )
)

b1_wape = (
    np.abs(actual - b1_pred).sum()
    / np.abs(actual).sum()
    * 100
)

# Bias = prediction - actual
b1_bias = np.mean(
    b1_pred - actual
)

# ------------------------------------------------------------
# Load R0 reference metrics
# ------------------------------------------------------------

summary_path = (
    NB5_ROOT / "Tables" /
    "Table_R0_R1_R2_July_Model_Selection.csv"
)

model_summary = pd.read_csv(summary_path)

r0_row = model_summary[
    model_summary["Model"] == "R0_CleanOnly"
].iloc[0]

r0_injected_wape = r0_row[
    "Controlled_Injected_WAPE"
]

# Improvement relative to R0 under injected failure
b1_improvement = (
    (r0_injected_wape - b1_wape)
    / r0_injected_wape
    * 100
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print("=" * 80)
print("B1 — PREVIOUS-DAY SUBSTITUTION RESULTS")
print("=" * 80)

print(f"Evaluation rows       : {len(actual):,}")
print(f"Prediction availability: 100.00%")

print("\nPerformance:")
print(f"WAPE                  : {b1_wape:.6f}%")
print(f"MAE                   : {b1_mae:.6f}")
print(f"RMSE                  : {b1_rmse:.6f}")
print(f"Bias                  : {b1_bias:.6f}")

print("\nReference:")
print(
    f"R0 Injected WAPE      : "
    f"{r0_injected_wape:.6f}%"
)

print(
    f"B1 improvement vs R0  : "
    f"{b1_improvement:.3f}%"
)

print("\nRaw negative predictions clipped to zero:")
print(
    f"{(b1_raw_pred < 0).sum():,}"
)

# ------------------------------------------------------------
# Save B1 predictions
# ------------------------------------------------------------

b1_output = baseline_eval[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume",
        "Missing_Lag_Count",
        "Missing_Lag_Pattern"
    ]
].copy()

b1_output["B1_PrevDay_Prediction"] = b1_pred

b1_output_path = (
    NB6_DATA /
    "B1_PreviousDay_Controlled_July_Predictions.parquet"
)

b1_output.to_parquet(
    b1_output_path,
    index=False
)

print("\nSaved:")
print(b1_output_path)

In [ ]:
# ============================================================
# CELL 16
# Evaluate B2 — Previous-Week Lag Substitution
# Matched-row comparison with R0
# ============================================================

# ------------------------------------------------------------
# Keep only rows where ALL required B2 lag inputs are available
# ------------------------------------------------------------

b2_mask = baseline_eval["B2_All_Lags_Available"].copy()

b2_data = baseline_eval.loc[b2_mask].copy()

print("=" * 80)
print("B2 — PREVIOUS-WEEK SUBSTITUTION")
print("=" * 80)

print(f"Total controlled rows : {len(baseline_eval):,}")
print(f"Available B2 rows     : {len(b2_data):,}")
print(
    f"Row availability      : "
    f"{100 * len(b2_data) / len(baseline_eval):.2f}%"
)

# ------------------------------------------------------------
# Build exact R0 model feature matrix
# ------------------------------------------------------------

X_b2 = pd.DataFrame({
    "Detector_Code": b2_data["Detector_Code"],
    "Interval_Index": b2_data["Interval_Index"],
    "DayOfWeek": b2_data["DayOfWeek"],
    "IsWeekend": b2_data["IsWeekend"],
    "Lag_1": b2_data["Lag_1_B2_PrevWeek"],
    "Lag_2": b2_data["Lag_2_B2_PrevWeek"],
    "Lag_4": b2_data["Lag_4_B2_PrevWeek"],
    "Lag_8": b2_data["Lag_8_B2_PrevWeek"],
    "Lag_96": b2_data["Lag_96_B2_PrevWeek"],
})

assert X_b2.isna().sum().sum() == 0, \
    "B2 feature matrix still contains NaNs."

# ------------------------------------------------------------
# Predict using ORIGINAL clean-only R0 model
# ------------------------------------------------------------

b2_raw_pred = r0_model.predict(X_b2)

# Same post-processing used in Notebook 5
b2_pred = np.maximum(b2_raw_pred, 0)

actual_b2 = b2_data["Target_Volume"].to_numpy()

# ------------------------------------------------------------
# Metric helper
# ------------------------------------------------------------

def calculate_metrics(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    wape = (
        np.abs(y_true - y_pred).sum()
        / np.abs(y_true).sum()
        * 100
    )

    bias = np.mean(y_pred - y_true)

    return {
        "WAPE": wape,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias
    }

b2_metrics = calculate_metrics(
    actual_b2,
    b2_pred
)

# ------------------------------------------------------------
# Get R0 injected predictions for EXACT SAME rows
# ------------------------------------------------------------

b2_compare = b2_data[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume"
    ]
].copy()

b2_compare["B2_PrevWeek_Prediction"] = b2_pred

b2_compare = b2_compare.merge(
    r0_saved[
        [
            "July_Validation_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Injected_Prediction"
        ]
    ],
    on=[
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

assert (
    b2_compare["R0_Injected_Prediction"].notna().all()
), "Missing matched R0 predictions."

matched_actual = b2_compare[
    "Target_Volume"
].to_numpy()

matched_r0_pred = b2_compare[
    "R0_Injected_Prediction"
].to_numpy()

r0_matched_metrics = calculate_metrics(
    matched_actual,
    matched_r0_pred
)

# ------------------------------------------------------------
# Improvement against MATCHED R0 subset
# ------------------------------------------------------------

b2_improvement_matched = (
    (
        r0_matched_metrics["WAPE"]
        - b2_metrics["WAPE"]
    )
    / r0_matched_metrics["WAPE"]
    * 100
)

# ------------------------------------------------------------
# Fully available events
# ------------------------------------------------------------

b2_full_events = (
    event_availability[
        "B2_Event_Fully_Available"
    ].sum()
)

total_events = len(event_availability)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nPerformance on available rows:")
print(f"B2 WAPE             : {b2_metrics['WAPE']:.6f}%")
print(f"B2 MAE              : {b2_metrics['MAE']:.6f}")
print(f"B2 RMSE             : {b2_metrics['RMSE']:.6f}")
print(f"B2 Bias             : {b2_metrics['Bias']:.6f}")

print("\nMatched R0 performance on SAME rows:")
print(
    f"R0 matched WAPE     : "
    f"{r0_matched_metrics['WAPE']:.6f}%"
)
print(
    f"R0 matched MAE      : "
    f"{r0_matched_metrics['MAE']:.6f}"
)
print(
    f"R0 matched RMSE     : "
    f"{r0_matched_metrics['RMSE']:.6f}"
)
print(
    f"R0 matched Bias     : "
    f"{r0_matched_metrics['Bias']:.6f}"
)

print(
    f"\nB2 improvement vs matched R0: "
    f"{b2_improvement_matched:.3f}%"
)

print(
    f"\nFully available events: "
    f"{b2_full_events:,}/{total_events:,} "
    f"({100*b2_full_events/total_events:.2f}%)"
)

print(
    f"Raw negative predictions clipped to zero: "
    f"{(b2_raw_pred < 0).sum():,}"
)

# ------------------------------------------------------------
# Save B2 predictions
# ------------------------------------------------------------

b2_output = baseline_eval[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume",
        "Missing_Lag_Count",
        "Missing_Lag_Pattern",
        "B2_All_Lags_Available"
    ]
].copy()

b2_output["B2_PrevWeek_Prediction"] = np.nan

b2_output.loc[
    b2_mask,
    "B2_PrevWeek_Prediction"
] = b2_pred

b2_output_path = (
    NB6_DATA /
    "B2_PreviousWeek_Controlled_July_Predictions.parquet"
)

b2_output.to_parquet(
    b2_output_path,
    index=False
)

print("\nSaved:")
print(b2_output_path)

In [ ]:
# ============================================================
# CELL 17
# Evaluate B3 — Seasonal-Naive Fallback
# ============================================================

# Seasonal naive forecast:
# prediction at time t = observed volume at t - 96 intervals
#                      = Lag_96_Injected
#
# IMPORTANT:
# If Lag_96 itself is unavailable because of the injected
# failure, the seasonal-naive forecast is considered unavailable.
# No artificial replacement is introduced.

b3_available_mask = (
    baseline_eval["Lag_96_Injected"].notna()
)

b3_data = baseline_eval.loc[
    b3_available_mask
].copy()

print("=" * 80)
print("B3 — SEASONAL-NAIVE FALLBACK")
print("=" * 80)

print(f"Total controlled rows : {len(baseline_eval):,}")
print(f"Available B3 rows     : {len(b3_data):,}")

print(
    f"Row availability      : "
    f"{100 * len(b3_data) / len(baseline_eval):.2f}%"
)

# ------------------------------------------------------------
# Seasonal-naive predictions
# ------------------------------------------------------------

b3_pred = (
    b3_data["Lag_96_Injected"]
    .to_numpy()
)

actual_b3 = (
    b3_data["Target_Volume"]
    .to_numpy()
)

# ------------------------------------------------------------
# B3 metrics
# ------------------------------------------------------------

b3_metrics = calculate_metrics(
    actual_b3,
    b3_pred
)

# ------------------------------------------------------------
# Matched R0 comparison
# Use EXACT SAME rows for fair comparison
# ------------------------------------------------------------

b3_compare = b3_data[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume"
    ]
].copy()

b3_compare["B3_SeasonalNaive_Prediction"] = b3_pred

b3_compare = b3_compare.merge(
    r0_saved[
        [
            "July_Validation_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Injected_Prediction"
        ]
    ],
    on=[
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

assert (
    b3_compare["R0_Injected_Prediction"]
    .notna()
    .all()
), "Missing matched R0 predictions."

matched_actual_b3 = (
    b3_compare["Target_Volume"]
    .to_numpy()
)

matched_r0_b3 = (
    b3_compare["R0_Injected_Prediction"]
    .to_numpy()
)

r0_b3_matched_metrics = calculate_metrics(
    matched_actual_b3,
    matched_r0_b3
)

# ------------------------------------------------------------
# Improvement versus matched R0
# ------------------------------------------------------------

b3_improvement_matched = (
    (
        r0_b3_matched_metrics["WAPE"]
        - b3_metrics["WAPE"]
    )
    / r0_b3_matched_metrics["WAPE"]
    * 100
)

# ------------------------------------------------------------
# Event-level availability
# ------------------------------------------------------------

b3_event_availability = (
    baseline_eval
    .assign(
        B3_Available=
        baseline_eval["Lag_96_Injected"].notna()
    )
    .groupby("July_Validation_Event_ID")
    .agg(
        B3_Event_Fully_Available=(
            "B3_Available", "all"
        ),
        B3_Event_Any_Available=(
            "B3_Available", "any"
        )
    )
    .reset_index()
)

full_events = (
    b3_event_availability[
        "B3_Event_Fully_Available"
    ].sum()
)

any_events = (
    b3_event_availability[
        "B3_Event_Any_Available"
    ].sum()
)

total_events = len(
    b3_event_availability
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nPerformance on available rows:")
print(
    f"B3 WAPE             : "
    f"{b3_metrics['WAPE']:.6f}%"
)
print(
    f"B3 MAE              : "
    f"{b3_metrics['MAE']:.6f}"
)
print(
    f"B3 RMSE             : "
    f"{b3_metrics['RMSE']:.6f}"
)
print(
    f"B3 Bias             : "
    f"{b3_metrics['Bias']:.6f}"
)

print("\nMatched R0 performance on SAME rows:")
print(
    f"R0 matched WAPE     : "
    f"{r0_b3_matched_metrics['WAPE']:.6f}%"
)
print(
    f"R0 matched MAE      : "
    f"{r0_b3_matched_metrics['MAE']:.6f}"
)
print(
    f"R0 matched RMSE     : "
    f"{r0_b3_matched_metrics['RMSE']:.6f}"
)
print(
    f"R0 matched Bias     : "
    f"{r0_b3_matched_metrics['Bias']:.6f}"
)

print(
    f"\nB3 improvement vs matched R0: "
    f"{b3_improvement_matched:.3f}%"
)

print("\nEvent-level availability:")

print(
    f"Events with B3 available for EVERY row: "
    f"{full_events:,}/{total_events:,} "
    f"({100 * full_events / total_events:.2f}%)"
)

print(
    f"Events with B3 available for AT LEAST one row: "
    f"{any_events:,}/{total_events:,} "
    f"({100 * any_events / total_events:.2f}%)"
)

# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------

b3_output = baseline_eval[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume",
        "Missing_Lag_Count",
        "Missing_Lag_Pattern"
    ]
].copy()

b3_output["B3_SeasonalNaive_Available"] = (
    baseline_eval["Lag_96_Injected"].notna()
)

b3_output[
    "B3_SeasonalNaive_Prediction"
] = baseline_eval["Lag_96_Injected"]

b3_output_path = (
    NB6_DATA /
    "B3_SeasonalNaive_Fallback_Controlled_July_Predictions.parquet"
)

b3_output.to_parquet(
    b3_output_path,
    index=False
)

print("\nSaved:")
print(b3_output_path)

In [ ]:
# ============================================================
# CELL 18
# Unified Practical Baseline Summary
# ============================================================

# ------------------------------------------------------------
# Load R1 and R2 stored predictions
# ------------------------------------------------------------

r1_saved = pd.read_parquet(
    NB5_ROOT / "Data" /
    "R1_Controlled_July_Predictions.parquet"
)

r2_saved = pd.read_parquet(
    NB5_ROOT / "Data" /
    "R2_Controlled_July_Predictions.parquet"
)

r1_saved["Timestamp"] = pd.to_datetime(r1_saved["Timestamp"])
r2_saved["Timestamp"] = pd.to_datetime(r2_saved["Timestamp"])

# ------------------------------------------------------------
# Existing R0/R1/R2 summary from Notebook 5
# ------------------------------------------------------------

existing_summary = pd.read_csv(
    NB5_ROOT / "Tables" /
    "Table_R0_R1_R2_July_Model_Selection.csv"
)

r0_stats = existing_summary[
    existing_summary["Model"] == "R0_CleanOnly"
].iloc[0]

r1_stats = existing_summary[
    existing_summary["Model"] == "R1_FailureAware"
].iloc[0]

r2_stats = existing_summary[
    existing_summary["Model"] == "R2_FailureAware_Indicators"
].iloc[0]

# ------------------------------------------------------------
# Construct unified table
# ------------------------------------------------------------

summary_rows = []

# R0
summary_rows.append({
    "Strategy": "R0 Clean-Only",
    "Strategy_Type": "Model",
    "Evaluation_Rows": len(baseline_eval),
    "Row_Availability_Percent": 100.0,
    "Fully_Available_Events": 581,
    "Event_Availability_Percent": 100.0,
    "WAPE": r0_stats["Controlled_Injected_WAPE"],
    "MAE": r0_stats["Injected_MAE"],
    "RMSE": r0_stats["Injected_RMSE"],
    "Bias": r0_stats["Injected_Bias"],
    "Comparison_Basis": "All controlled rows"
})

# R1
summary_rows.append({
    "Strategy": "R1 Failure-Aware",
    "Strategy_Type": "Model",
    "Evaluation_Rows": len(baseline_eval),
    "Row_Availability_Percent": 100.0,
    "Fully_Available_Events": 581,
    "Event_Availability_Percent": 100.0,
    "WAPE": r1_stats["Controlled_Injected_WAPE"],
    "MAE": r1_stats["Injected_MAE"],
    "RMSE": r1_stats["Injected_RMSE"],
    "Bias": r1_stats["Injected_Bias"],
    "Comparison_Basis": "All controlled rows"
})

# R2
summary_rows.append({
    "Strategy": "R2 Failure-Aware + Indicators",
    "Strategy_Type": "Model",
    "Evaluation_Rows": len(baseline_eval),
    "Row_Availability_Percent": 100.0,
    "Fully_Available_Events": 581,
    "Event_Availability_Percent": 100.0,
    "WAPE": r2_stats["Controlled_Injected_WAPE"],
    "MAE": r2_stats["Injected_MAE"],
    "RMSE": r2_stats["Injected_RMSE"],
    "Bias": r2_stats["Injected_Bias"],
    "Comparison_Basis": "All controlled rows"
})

# B1
summary_rows.append({
    "Strategy": "B1 Previous-Day Imputation",
    "Strategy_Type": "Practical baseline",
    "Evaluation_Rows": len(baseline_eval),
    "Row_Availability_Percent": 100.0,
    "Fully_Available_Events": 581,
    "Event_Availability_Percent": 100.0,
    "WAPE": b1_wape,
    "MAE": b1_mae,
    "RMSE": b1_rmse,
    "Bias": b1_bias,
    "Comparison_Basis": "All controlled rows"
})

# B2
summary_rows.append({
    "Strategy": "B2 Previous-Week Imputation",
    "Strategy_Type": "Practical baseline",
    "Evaluation_Rows": len(b2_data),
    "Row_Availability_Percent":
        100 * len(b2_data) / len(baseline_eval),
    "Fully_Available_Events": int(b2_full_events),
    "Event_Availability_Percent":
        100 * b2_full_events / total_events,
    "WAPE": b2_metrics["WAPE"],
    "MAE": b2_metrics["MAE"],
    "RMSE": b2_metrics["RMSE"],
    "Bias": b2_metrics["Bias"],
    "Comparison_Basis": "Available rows only"
})

# B3
summary_rows.append({
    "Strategy": "B3 Seasonal-Naive Fallback",
    "Strategy_Type": "Practical baseline",
    "Evaluation_Rows": len(b3_data),
    "Row_Availability_Percent":
        100 * len(b3_data) / len(baseline_eval),
    "Fully_Available_Events": int(full_events),
    "Event_Availability_Percent":
        100 * full_events / total_events,
    "WAPE": b3_metrics["WAPE"],
    "MAE": b3_metrics["MAE"],
    "RMSE": b3_metrics["RMSE"],
    "Bias": b3_metrics["Bias"],
    "Comparison_Basis": "Available rows only"
})

practical_summary = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Round display copy only
# ------------------------------------------------------------

display_summary = practical_summary.copy()

for c in [
    "Row_Availability_Percent",
    "Event_Availability_Percent",
    "WAPE",
    "MAE",
    "RMSE",
    "Bias"
]:
    display_summary[c] = display_summary[c].round(3)

print("=" * 110)
print("UNIFIED ROBUSTNESS AND PRACTICAL-BASELINE SUMMARY")
print("=" * 110)

display(display_summary)

# ------------------------------------------------------------
# Direct full-coverage comparison
# Only methods evaluable on all 12,124 rows
# ------------------------------------------------------------

full_coverage = practical_summary[
    practical_summary["Row_Availability_Percent"] == 100.0
].copy()

print("\n" + "=" * 110)
print("FULL-COVERAGE METHODS — DIRECTLY COMPARABLE ON ALL 12,124 ROWS")
print("=" * 110)

display(
    full_coverage[
        [
            "Strategy",
            "WAPE",
            "MAE",
            "RMSE",
            "Bias",
            "Row_Availability_Percent"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

summary_output_path = (
    NB6_TABLES /
    "Table_Practical_Baselines_and_FailureAware_Models.csv"
)

practical_summary.to_csv(
    summary_output_path,
    index=False
)

print("\nSaved:")
print(summary_output_path)

In [ ]:
# ============================================================
# CELL 19
# Matched-Sample Fair Comparison for B2 and B3
# ============================================================

# ------------------------------------------------------------
# Prepare unified stored predictions
# ------------------------------------------------------------

prediction_keys = [
    "July_Validation_Event_ID",
    "Detector_Code",
    "Timestamp"
]

# Keep only required columns
r0_match = r0_saved[
    prediction_keys + ["R0_Injected_Prediction"]
].copy()

r1_match = r1_saved[
    prediction_keys + ["R1_Injected_Prediction"]
].copy()

r2_match = r2_saved[
    prediction_keys + ["R2_Injected_Prediction"]
].copy()

# B1 predictions already computed on all rows
b1_match = b1_output[
    prediction_keys + ["B1_PrevDay_Prediction"]
].copy()

# Build common evaluation frame
matched_all = baseline_eval[
    prediction_keys + ["Target_Volume"]
].copy()

matched_all = matched_all.merge(
    r0_match,
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

matched_all = matched_all.merge(
    r1_match,
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

matched_all = matched_all.merge(
    r2_match,
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

matched_all = matched_all.merge(
    b1_match,
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

# Add B2 and B3 predictions
matched_all = matched_all.merge(
    b2_output[
        prediction_keys + ["B2_PrevWeek_Prediction"]
    ],
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

matched_all = matched_all.merge(
    b3_output[
        prediction_keys + ["B3_SeasonalNaive_Prediction"]
    ],
    on=prediction_keys,
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def evaluate_subset(df, prediction_col):

    valid = df[
        prediction_col
    ].notna()

    y_true = df.loc[
        valid,
        "Target_Volume"
    ].to_numpy()

    y_pred = df.loc[
        valid,
        prediction_col
    ].to_numpy()

    m = calculate_metrics(
        y_true,
        y_pred
    )

    return {
        "Rows": len(y_true),
        "WAPE": m["WAPE"],
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "Bias": m["Bias"]
    }


# ------------------------------------------------------------
# B2 matched subset
# ------------------------------------------------------------

b2_subset = matched_all[
    matched_all["B2_PrevWeek_Prediction"].notna()
].copy()

b2_rows = []

for label, col in [
    ("R0 Clean-Only", "R0_Injected_Prediction"),
    ("R1 Failure-Aware", "R1_Injected_Prediction"),
    ("R2 Failure-Aware + Indicators", "R2_Injected_Prediction"),
    ("B1 Previous-Day Imputation", "B1_PrevDay_Prediction"),
    ("B2 Previous-Week Imputation", "B2_PrevWeek_Prediction"),
]:

    result = evaluate_subset(
        b2_subset,
        col
    )

    result["Strategy"] = label
    b2_rows.append(result)

b2_matched_summary = pd.DataFrame(
    b2_rows
)[
    [
        "Strategy",
        "Rows",
        "WAPE",
        "MAE",
        "RMSE",
        "Bias"
    ]
]


# ------------------------------------------------------------
# B3 matched subset
# ------------------------------------------------------------

b3_subset = matched_all[
    matched_all["B3_SeasonalNaive_Prediction"].notna()
].copy()

b3_rows = []

for label, col in [
    ("R0 Clean-Only", "R0_Injected_Prediction"),
    ("R1 Failure-Aware", "R1_Injected_Prediction"),
    ("R2 Failure-Aware + Indicators", "R2_Injected_Prediction"),
    ("B1 Previous-Day Imputation", "B1_PrevDay_Prediction"),
    ("B3 Seasonal-Naive Fallback", "B3_SeasonalNaive_Prediction"),
]:

    result = evaluate_subset(
        b3_subset,
        col
    )

    result["Strategy"] = label
    b3_rows.append(result)

b3_matched_summary = pd.DataFrame(
    b3_rows
)[
    [
        "Strategy",
        "Rows",
        "WAPE",
        "MAE",
        "RMSE",
        "Bias"
    ]
]


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print(
    "MATCHED COMPARISON — B2 AVAILABLE ROWS "
    f"({len(b2_subset):,} rows)"
)
print("=" * 100)

display(
    b2_matched_summary.round(4)
)

print("\n" + "=" * 100)
print(
    "MATCHED COMPARISON — B3 AVAILABLE ROWS "
    f"({len(b3_subset):,} rows)"
)
print("=" * 100)

display(
    b3_matched_summary.round(4)
)

# ------------------------------------------------------------
# Calculate direct differences vs R1
# ------------------------------------------------------------

b2_r1_wape = b2_matched_summary.loc[
    b2_matched_summary["Strategy"] == "R1 Failure-Aware",
    "WAPE"
].iloc[0]

b2_b2_wape = b2_matched_summary.loc[
    b2_matched_summary["Strategy"] == "B2 Previous-Week Imputation",
    "WAPE"
].iloc[0]

b3_r1_wape = b3_matched_summary.loc[
    b3_matched_summary["Strategy"] == "R1 Failure-Aware",
    "WAPE"
].iloc[0]

b3_b3_wape = b3_matched_summary.loc[
    b3_matched_summary["Strategy"] == "B3 Seasonal-Naive Fallback",
    "WAPE"
].iloc[0]

print("\n" + "=" * 100)
print("DIRECT WAPE DIFFERENCES VS R1 ON IDENTICAL ROWS")
print("=" * 100)

print(
    f"B2 minus R1 WAPE : "
    f"{b2_b2_wape - b2_r1_wape:+.4f} percentage points"
)

print(
    f"B3 minus R1 WAPE : "
    f"{b3_b3_wape - b3_r1_wape:+.4f} percentage points"
)

# ------------------------------------------------------------
# Save matched summaries
# ------------------------------------------------------------

b2_matched_path = (
    NB6_TABLES /
    "Table_B2_Matched_Sample_Comparison.csv"
)

b3_matched_path = (
    NB6_TABLES /
    "Table_B3_Matched_Sample_Comparison.csv"
)

b2_matched_summary.to_csv(
    b2_matched_path,
    index=False
)

b3_matched_summary.to_csv(
    b3_matched_path,
    index=False
)

print("\nSaved:")
print(b2_matched_path)
print(b3_matched_path)

In [ ]:
# ============================================================
# CELL 20
# Event-Level Paired Comparison
# R1 Failure-Aware vs B1 Previous-Day Imputation
# ============================================================

# Build event-level comparison dataset
event_compare = matched_all[
    [
        "July_Validation_Event_ID",
        "Target_Volume",
        "R0_Injected_Prediction",
        "R1_Injected_Prediction",
        "R2_Injected_Prediction",
        "B1_PrevDay_Prediction"
    ]
].copy()


def event_metrics(group, pred_col):

    y_true = group["Target_Volume"].to_numpy()
    y_pred = group[pred_col].to_numpy()

    abs_error = np.abs(y_true - y_pred)

    denom = np.abs(y_true).sum()

    # Event WAPE
    if denom > 0:
        wape = abs_error.sum() / denom * 100
    else:
        wape = np.nan

    mae = abs_error.mean()

    rmse = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

    bias = np.mean(
        y_pred - y_true
    )

    return pd.Series({
        "WAPE": wape,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias
    })


# ------------------------------------------------------------
# Calculate event-level metrics
# ------------------------------------------------------------

event_tables = {}

prediction_map = {
    "R0": "R0_Injected_Prediction",
    "R1": "R1_Injected_Prediction",
    "R2": "R2_Injected_Prediction",
    "B1": "B1_PrevDay_Prediction"
}

for label, pred_col in prediction_map.items():

    tmp = (
        event_compare
        .groupby("July_Validation_Event_ID")
        .apply(
            lambda g: event_metrics(g, pred_col)
        )
        .reset_index()
    )

    tmp = tmp.rename(
        columns={
            "WAPE": f"{label}_WAPE",
            "MAE": f"{label}_MAE",
            "RMSE": f"{label}_RMSE",
            "Bias": f"{label}_Bias"
        }
    )

    event_tables[label] = tmp


# Merge event-level results
event_results = event_tables["R0"]

for label in ["R1", "R2", "B1"]:
    event_results = event_results.merge(
        event_tables[label],
        on="July_Validation_Event_ID",
        how="inner",
        validate="one_to_one"
    )


print("=" * 95)
print("EVENT-LEVEL PAIRED COMPARISON")
print("=" * 95)

print(f"Events evaluated: {len(event_results):,}")


# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

summary_event_rows = []

for label in ["R0", "R1", "R2", "B1"]:

    wape = event_results[f"{label}_WAPE"]

    summary_event_rows.append({
        "Strategy": label,
        "Mean_Event_WAPE": wape.mean(),
        "Median_Event_WAPE": wape.median(),
        "P25_Event_WAPE": wape.quantile(0.25),
        "P75_Event_WAPE": wape.quantile(0.75),
        "P95_Event_WAPE": wape.quantile(0.95)
    })

event_summary = pd.DataFrame(summary_event_rows)

print("\nEvent-level WAPE summary:")
display(
    event_summary.round(4)
)


# ------------------------------------------------------------
# Direct paired comparison: B1 vs R1
# ------------------------------------------------------------

valid_pair = (
    event_results["R1_WAPE"].notna()
    & event_results["B1_WAPE"].notna()
)

paired = event_results.loc[
    valid_pair
].copy()

paired["B1_minus_R1_WAPE"] = (
    paired["B1_WAPE"]
    - paired["R1_WAPE"]
)

b1_better = (
    paired["B1_WAPE"]
    < paired["R1_WAPE"]
).sum()

r1_better = (
    paired["R1_WAPE"]
    < paired["B1_WAPE"]
).sum()

ties = (
    np.isclose(
        paired["B1_WAPE"],
        paired["R1_WAPE"]
    )
).sum()


print("\n" + "=" * 95)
print("B1 vs R1 — EVENT-LEVEL RESULTS")
print("=" * 95)

print(f"Paired events              : {len(paired):,}")

print(
    f"B1 lower WAPE             : "
    f"{b1_better:,} "
    f"({100*b1_better/len(paired):.2f}%)"
)

print(
    f"R1 lower WAPE             : "
    f"{r1_better:,} "
    f"({100*r1_better/len(paired):.2f}%)"
)

print(
    f"Ties                      : "
    f"{ties:,}"
)

print(
    f"Median B1 - R1 WAPE diff  : "
    f"{paired['B1_minus_R1_WAPE'].median():+.4f} pp"
)

print(
    f"Mean B1 - R1 WAPE diff    : "
    f"{paired['B1_minus_R1_WAPE'].mean():+.4f} pp"
)


# ------------------------------------------------------------
# Wilcoxon paired signed-rank test
# ------------------------------------------------------------

from scipy.stats import wilcoxon

nonzero_diff = paired[
    ~np.isclose(
        paired["B1_minus_R1_WAPE"],
        0.0
    )
]

if len(nonzero_diff) > 0:

    stat, p_value = wilcoxon(
        nonzero_diff["B1_WAPE"],
        nonzero_diff["R1_WAPE"],
        alternative="two-sided"
    )

    print("\nPaired Wilcoxon signed-rank test:")
    print(f"Statistic                 : {stat:.6f}")
    print(f"P-value                   : {p_value:.6e}")

else:
    print("\nWilcoxon test not applicable: all differences are zero.")


# ------------------------------------------------------------
# Save event-level results
# ------------------------------------------------------------

event_results_path = (
    NB6_DATA /
    "Event_Level_R0_R1_R2_B1_Comparison.parquet"
)

event_summary_path = (
    NB6_TABLES /
    "Table_Event_Level_Practical_Baseline_Summary.csv"
)

event_results.to_parquet(
    event_results_path,
    index=False
)

event_summary.to_csv(
    event_summary_path,
    index=False
)

print("\nSaved:")
print(event_results_path)
print(event_summary_path)

In [ ]:
# ============================================================
# CELL 21
# Performance by Failure Duration Group
# R0 vs R1 vs B1
# ============================================================

# ------------------------------------------------------------
# Add failure-duration information to matched predictions
# ------------------------------------------------------------

duration_info = baseline_eval[
    [
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Failure_Duration_Group",
        "Failure_Duration_Hours"
    ]
].copy()

duration_compare = matched_all.merge(
    duration_info,
    on=[
        "July_Validation_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Helper for aggregated group metrics
# ------------------------------------------------------------

def grouped_metrics(df, pred_col):

    y_true = df["Target_Volume"].to_numpy()
    y_pred = df[pred_col].to_numpy()

    mae = np.mean(
        np.abs(y_true - y_pred)
    )

    rmse = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

    denom = np.abs(y_true).sum()

    wape = (
        np.abs(y_true - y_pred).sum()
        / denom
        * 100
        if denom > 0
        else np.nan
    )

    bias = np.mean(
        y_pred - y_true
    )

    return {
        "Rows": len(df),
        "WAPE": wape,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias
    }


# ------------------------------------------------------------
# Preferred duration order
# ------------------------------------------------------------

duration_order = [
    "15 min",
    "30–60 min",
    "1.25–2 h",
    "2.25–6 h",
    "6.25–12 h",
    "12.25–23.75 h",
    "24 h"
]

results = []

for group in duration_order:

    subset = duration_compare[
        duration_compare["Failure_Duration_Group"] == group
    ].copy()

    if len(subset) == 0:
        continue

    # Number of unique failure events in this group
    n_events = subset[
        "July_Validation_Event_ID"
    ].nunique()

    for strategy, pred_col in [
        ("R0 Clean-Only", "R0_Injected_Prediction"),
        ("R1 Failure-Aware", "R1_Injected_Prediction"),
        ("B1 Previous-Day Imputation", "B1_PrevDay_Prediction")
    ]:

        m = grouped_metrics(
            subset,
            pred_col
        )

        results.append({
            "Failure_Duration_Group": group,
            "Strategy": strategy,
            "Events": n_events,
            "Rows": m["Rows"],
            "WAPE": m["WAPE"],
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "Bias": m["Bias"]
        })


duration_summary = pd.DataFrame(results)

# ------------------------------------------------------------
# Display full table
# ------------------------------------------------------------

print("=" * 110)
print("PERFORMANCE BY FAILURE DURATION GROUP")
print("=" * 110)

display(
    duration_summary.round(4)
)

# ------------------------------------------------------------
# Direct B1 vs R1 WAPE comparison
# ------------------------------------------------------------

pivot_wape = duration_summary.pivot(
    index="Failure_Duration_Group",
    columns="Strategy",
    values="WAPE"
).reset_index()

pivot_wape["B1_minus_R1_WAPE_pp"] = (
    pivot_wape["B1 Previous-Day Imputation"]
    - pivot_wape["R1 Failure-Aware"]
)

pivot_wape["B1_Relative_Improvement_vs_R1_Percent"] = (
    (
        pivot_wape["R1 Failure-Aware"]
        - pivot_wape["B1 Previous-Day Imputation"]
    )
    / pivot_wape["R1 Failure-Aware"]
    * 100
)

# Restore duration order
pivot_wape["Failure_Duration_Group"] = pd.Categorical(
    pivot_wape["Failure_Duration_Group"],
    categories=duration_order,
    ordered=True
)

pivot_wape = pivot_wape.sort_values(
    "Failure_Duration_Group"
)

print("\n" + "=" * 110)
print("DIRECT B1 vs R1 COMPARISON BY FAILURE DURATION")
print("=" * 110)

display(
    pivot_wape[
        [
            "Failure_Duration_Group",
            "R1 Failure-Aware",
            "B1 Previous-Day Imputation",
            "B1_minus_R1_WAPE_pp",
            "B1_Relative_Improvement_vs_R1_Percent"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

duration_summary_path = (
    NB6_TABLES /
    "Table_Performance_by_Failure_Duration.csv"
)

duration_comparison_path = (
    NB6_TABLES /
    "Table_B1_vs_R1_by_Failure_Duration.csv"
)

duration_summary.to_csv(
    duration_summary_path,
    index=False
)

pivot_wape.to_csv(
    duration_comparison_path,
    index=False
)

print("\nSaved:")
print(duration_summary_path)
print(duration_comparison_path)

In [ ]:
# ============================================================
# CELL 22
# Final Manuscript-Ready Practical Baseline Comparison Table
# ============================================================

# ------------------------------------------------------------
# Build compact publication table
# ------------------------------------------------------------

final_table = pd.DataFrame([
    {
        "Method": "R0 Clean-only",
        "Mitigation": "None",
        "Evaluation_Rows": 12124,
        "Row_Availability_Percent": 100.0,
        "WAPE_Percent": r0_stats["Controlled_Injected_WAPE"],
        "MAE": r0_stats["Injected_MAE"],
        "RMSE": r0_stats["Injected_RMSE"],
        "Bias": r0_stats["Injected_Bias"],
        "Evaluation_Note": "Full controlled set"
    },
    {
        "Method": "R1 Failure-aware",
        "Mitigation": "Failure-aware training",
        "Evaluation_Rows": 12124,
        "Row_Availability_Percent": 100.0,
        "WAPE_Percent": r1_stats["Controlled_Injected_WAPE"],
        "MAE": r1_stats["Injected_MAE"],
        "RMSE": r1_stats["Injected_RMSE"],
        "Bias": r1_stats["Injected_Bias"],
        "Evaluation_Note": "Full controlled set"
    },
    {
        "Method": "R2 Failure-aware + indicators",
        "Mitigation": "Failure-aware training + indicators",
        "Evaluation_Rows": 12124,
        "Row_Availability_Percent": 100.0,
        "WAPE_Percent": r2_stats["Controlled_Injected_WAPE"],
        "MAE": r2_stats["Injected_MAE"],
        "RMSE": r2_stats["Injected_RMSE"],
        "Bias": r2_stats["Injected_Bias"],
        "Evaluation_Note": "Full controlled set"
    },
    {
        "Method": "B1 Previous-day imputation",
        "Mitigation": "Replace missing lag with same interval from previous day",
        "Evaluation_Rows": 12124,
        "Row_Availability_Percent": 100.0,
        "WAPE_Percent": b1_wape,
        "MAE": b1_mae,
        "RMSE": b1_rmse,
        "Bias": b1_bias,
        "Evaluation_Note": "Full controlled set"
    },
    {
        "Method": "B2 Previous-week imputation",
        "Mitigation": "Replace missing lag with same interval from previous week",
        "Evaluation_Rows": len(b2_data),
        "Row_Availability_Percent":
            100 * len(b2_data) / len(baseline_eval),
        "WAPE_Percent": b2_metrics["WAPE"],
        "MAE": b2_metrics["MAE"],
        "RMSE": b2_metrics["RMSE"],
        "Bias": b2_metrics["Bias"],
        "Evaluation_Note": "Available rows only"
    },
    {
        "Method": "B3 Seasonal-naive fallback",
        "Mitigation": "Use observed volume from same interval previous day",
        "Evaluation_Rows": len(b3_data),
        "Row_Availability_Percent":
            100 * len(b3_data) / len(baseline_eval),
        "WAPE_Percent": b3_metrics["WAPE"],
        "MAE": b3_metrics["MAE"],
        "RMSE": b3_metrics["RMSE"],
        "Bias": b3_metrics["Bias"],
        "Evaluation_Note": "Available rows only"
    }
])

# ------------------------------------------------------------
# Add improvement vs R0 only for methods evaluated on full set
# ------------------------------------------------------------

final_table["WAPE_Improvement_vs_R0_Percent"] = np.nan

full_mask = (
    final_table["Row_Availability_Percent"] == 100.0
)

final_table.loc[
    full_mask,
    "WAPE_Improvement_vs_R0_Percent"
] = (
    (
        r0_stats["Controlled_Injected_WAPE"]
        - final_table.loc[full_mask, "WAPE_Percent"]
    )
    / r0_stats["Controlled_Injected_WAPE"]
    * 100
)

# ------------------------------------------------------------
# Round display copy
# ------------------------------------------------------------

final_display = final_table.copy()

round_cols = [
    "Row_Availability_Percent",
    "WAPE_Percent",
    "MAE",
    "RMSE",
    "Bias",
    "WAPE_Improvement_vs_R0_Percent"
]

for col in round_cols:
    final_display[col] = final_display[col].round(2)

print("=" * 120)
print("FINAL PRACTICAL BASELINE COMPARISON — MANUSCRIPT TABLE")
print("=" * 120)

display(final_display)

# ------------------------------------------------------------
# Key full-coverage findings
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("KEY FULL-COVERAGE RESULTS")
print("=" * 120)

print(
    f"R0 WAPE : "
    f"{r0_stats['Controlled_Injected_WAPE']:.2f}%"
)

print(
    f"R1 WAPE : "
    f"{r1_stats['Controlled_Injected_WAPE']:.2f}%"
)

print(
    f"B1 WAPE : "
    f"{b1_wape:.2f}%"
)

print(
    f"\nB1 improvement vs R0 : "
    f"{b1_improvement:.2f}%"
)

print(
    f"B1 WAPE difference vs R1 : "
    f"{b1_wape - r1_stats['Controlled_Injected_WAPE']:+.2f} pp"
)

print(
    f"B1 MAE difference vs R1  : "
    f"{b1_mae - r1_stats['Injected_MAE']:+.2f}"
)

print(
    f"B1 RMSE difference vs R1 : "
    f"{b1_rmse - r1_stats['Injected_RMSE']:+.2f}"
)

# ------------------------------------------------------------
# Availability notes for partial baselines
# ------------------------------------------------------------

print("\nPartial-coverage baselines:")

print(
    f"B2 availability : "
    f"{100 * len(b2_data) / len(baseline_eval):.2f}%"
)

print(
    f"B3 availability : "
    f"{100 * len(b3_data) / len(baseline_eval):.2f}%"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

final_table_path = (
    NB6_TABLES /
    "Table_Final_Practical_Baseline_Comparison.csv"
)

final_table.to_csv(
    final_table_path,
    index=False
)

print("\nSaved:")
print(final_table_path)

In [ ]:
# ============================================================
# CELL 23
# Manuscript-Ready Figure:
# Robustness WAPE with Prediction Availability
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

plot_df = final_table.copy()

labels = [
    "R0\nClean-only",
    "R1\nFailure-aware",
    "R2\nFailure-aware\n+ indicators",
    "B1\nPrevious-day\nimputation",
    "B2\nPrevious-week\nimputation",
    "B3\nSeasonal-naive\nfallback"
]

wape_values = plot_df["WAPE_Percent"].to_numpy()

availability_values = (
    plot_df["Row_Availability_Percent"].to_numpy()
)

x = np.arange(len(labels))

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 6.5)
)

bars = ax.bar(
    x,
    wape_values,
    width=0.68,
    edgecolor="black",
    linewidth=0.8
)

# ------------------------------------------------------------
# Add WAPE labels and availability
# ------------------------------------------------------------

for i, bar in enumerate(bars):

    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.8,
        f"{height:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height / 2,
        f"Availability\n{availability_values[i]:.1f}%",
        ha="center",
        va="center",
        fontsize=9
    )

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_ylabel(
    "WAPE (%)",
    fontsize=12
)

ax.set_xlabel(
    "Forecasting / Mitigation Strategy",
    fontsize=12
)

ax.set_xticks(x)
ax.set_xticklabels(
    labels,
    fontsize=10
)

ax.set_title(
    "Forecast Robustness under Controlled Detector Failures",
    fontsize=14,
    pad=14
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.set_ylim(
    0,
    max(wape_values) * 1.18
)

# Reference line at R1
ax.axhline(
    y=r1_stats["Controlled_Injected_WAPE"],
    linestyle="--",
    linewidth=1.2,
    alpha=0.7
)

ax.text(
    len(labels) - 0.55,
    r1_stats["Controlled_Injected_WAPE"] + 0.7,
    "R1 WAPE",
    ha="right",
    fontsize=9
)

# ------------------------------------------------------------
# Footnote
# ------------------------------------------------------------

fig.text(
    0.5,
    0.01,
    "Note: R0, R1, R2, and B1 are evaluated on all 12,124 controlled rows. "
    "B2 and B3 metrics are calculated only where the corresponding fallback "
    "information is available.",
    ha="center",
    fontsize=9,
    wrap=True
)

plt.tight_layout(
    rect=[0, 0.06, 1, 1]
)

# ------------------------------------------------------------
# Save high-resolution outputs
# ------------------------------------------------------------

png_path = (
    NB6_FIGURES /
    "Figure_Practical_Baseline_Robustness_WAPE.png"
)

pdf_path = (
    NB6_FIGURES /
    "Figure_Practical_Baseline_Robustness_WAPE.pdf"
)

plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("=" * 90)
print("FIGURE SAVED")
print("=" * 90)

print("PNG:")
print(png_path)

print("\nPDF:")
print(pdf_path)

In [ ]:
# ============================================================
# CELL 24
# Manuscript-Ready Figure:
# Robustness by Failure Duration
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

duration_order = [
    "15 min",
    "30–60 min",
    "1.25–2 h",
    "2.25–6 h",
    "6.25–12 h",
    "12.25–23.75 h",
    "24 h"
]

# Event counts from duration summary
event_counts = (
    duration_summary[
        duration_summary["Strategy"] == "R0 Clean-Only"
    ]
    .set_index("Failure_Duration_Group")["Events"]
    .reindex(duration_order)
)

# Pivot WAPE values
duration_plot = (
    duration_summary
    .pivot(
        index="Failure_Duration_Group",
        columns="Strategy",
        values="WAPE"
    )
    .reindex(duration_order)
)

x = np.arange(len(duration_order))

# X-axis labels with number of events
x_labels = [
    f"{group}\n(n={int(event_counts.loc[group])})"
    for group in duration_order
]

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 6.5)
)

ax.plot(
    x,
    duration_plot["R0 Clean-Only"],
    marker="o",
    linewidth=2,
    markersize=7,
    label="R0 Clean-only"
)

ax.plot(
    x,
    duration_plot["R1 Failure-Aware"],
    marker="s",
    linewidth=2,
    markersize=7,
    label="R1 Failure-aware"
)

ax.plot(
    x,
    duration_plot["B1 Previous-Day Imputation"],
    marker="^",
    linewidth=2,
    markersize=7,
    label="B1 Previous-day imputation"
)

# ------------------------------------------------------------
# Add WAPE values for R1 and B1
# ------------------------------------------------------------

for i in range(len(x)):

    r1_value = duration_plot[
        "R1 Failure-Aware"
    ].iloc[i]

    b1_value = duration_plot[
        "B1 Previous-Day Imputation"
    ].iloc[i]

    ax.annotate(
        f"{r1_value:.1f}",
        (x[i], r1_value),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8
    )

    ax.annotate(
        f"{b1_value:.1f}",
        (x[i], b1_value),
        textcoords="offset points",
        xytext=(0, -14),
        ha="center",
        fontsize=8
    )

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    x_labels,
    fontsize=9
)

ax.set_ylabel(
    "WAPE (%)",
    fontsize=12
)

ax.set_xlabel(
    "Injected Failure Duration",
    fontsize=12
)

ax.set_title(
    "Forecast Robustness across Detector-Failure Durations",
    fontsize=14,
    pad=14
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.legend(
    frameon=False,
    fontsize=10
)

ax.set_ylim(
    0,
    max(
        duration_plot[
            "R0 Clean-Only"
        ].max() * 1.12,
        30
    )
)

# ------------------------------------------------------------
# Footnote
# ------------------------------------------------------------

fig.text(
    0.5,
    0.01,
    "Values denote aggregated WAPE within each failure-duration group; "
    "n indicates the number of controlled failure events.",
    ha="center",
    fontsize=9
)

plt.tight_layout(
    rect=[0, 0.05, 1, 1]
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

png_path_duration = (
    NB6_FIGURES /
    "Figure_Robustness_by_Failure_Duration.png"
)

pdf_path_duration = (
    NB6_FIGURES /
    "Figure_Robustness_by_Failure_Duration.pdf"
)

plt.savefig(
    png_path_duration,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path_duration,
    bbox_inches="tight"
)

plt.show()

print("=" * 90)
print("FAILURE-DURATION FIGURE SAVED")
print("=" * 90)

print("PNG:")
print(png_path_duration)

print("\nPDF:")
print(pdf_path_duration)

In [ ]:
# ============================================================
# CELL 25
# Final Notebook 6 Audit and Key Results Export
# ============================================================

import json
from pathlib import Path

print("=" * 100)
print("NOTEBOOK 6 — FINAL AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Core study counts
# ------------------------------------------------------------

n_rows = len(baseline_eval)
n_events = baseline_eval["July_Validation_Event_ID"].nunique()
n_detectors = baseline_eval["Detector_Code"].nunique()

# ------------------------------------------------------------
# Key numerical findings
# ------------------------------------------------------------

key_results = {
    "Notebook": "Notebook 6 – Practical Missing-Input Baselines and Robustness Benchmarking",

    "Controlled_Validation_Rows": int(n_rows),
    "Controlled_Validation_Events": int(n_events),
    "Controlled_Validation_Detectors": int(n_detectors),

    # R0
    "R0_Injected_WAPE": float(
        r0_stats["Controlled_Injected_WAPE"]
    ),

    # R1
    "R1_Injected_WAPE": float(
        r1_stats["Controlled_Injected_WAPE"]
    ),

    # R2
    "R2_Injected_WAPE": float(
        r2_stats["Controlled_Injected_WAPE"]
    ),

    # B1
    "B1_PreviousDay_WAPE": float(b1_wape),
    "B1_PreviousDay_MAE": float(b1_mae),
    "B1_PreviousDay_RMSE": float(b1_rmse),
    "B1_PreviousDay_Bias": float(b1_bias),
    "B1_Row_Availability_Percent": 100.0,
    "B1_Event_Availability_Percent": 100.0,
    "B1_Improvement_vs_R0_Percent": float(
        b1_improvement
    ),

    # B2
    "B2_PreviousWeek_WAPE_AvailableRows": float(
        b2_metrics["WAPE"]
    ),
    "B2_Row_Availability_Percent": float(
        100 * len(b2_data) / len(baseline_eval)
    ),
    "B2_Event_Availability_Percent": float(
        100 * b2_full_events / total_events
    ),
    "B2_Matched_Improvement_vs_R0_Percent": float(
        b2_improvement_matched
    ),

    # B3
    "B3_SeasonalNaive_WAPE_AvailableRows": float(
        b3_metrics["WAPE"]
    ),
    "B3_Row_Availability_Percent": float(
        100 * len(b3_data) / len(baseline_eval)
    ),
    "B3_FullyAvailable_Event_Percent": float(
        100 * full_events / total_events
    ),
    "B3_Matched_Improvement_vs_R0_Percent": float(
        b3_improvement_matched
    ),

    # Direct B1 vs R1
    "B1_minus_R1_WAPE_pp_FullSet": float(
        b1_wape
        - r1_stats["Controlled_Injected_WAPE"]
    ),

    "B1_EventLevel_Better_Events": int(b1_better),
    "B1_EventLevel_Paired_Events": int(len(paired)),
    "B1_EventLevel_Better_Percent": float(
        100 * b1_better / len(paired)
    ),
    "Median_B1_minus_R1_Event_WAPE_pp": float(
        paired["B1_minus_R1_WAPE"].median()
    ),
    "Wilcoxon_B1_vs_R1_P_Value": float(p_value),
}

# ------------------------------------------------------------
# Print concise final findings
# ------------------------------------------------------------

print(f"\nControlled rows      : {n_rows:,}")
print(f"Failure events       : {n_events:,}")
print(f"Detectors            : {n_detectors:,}")

print("\n" + "-" * 100)
print("FULL-COVERAGE METHODS")
print("-" * 100)

print(
    f"R0 Clean-only WAPE             : "
    f"{r0_stats['Controlled_Injected_WAPE']:.2f}%"
)

print(
    f"R1 Failure-aware WAPE          : "
    f"{r1_stats['Controlled_Injected_WAPE']:.2f}%"
)

print(
    f"R2 Failure-aware + indicators  : "
    f"{r2_stats['Controlled_Injected_WAPE']:.2f}%"
)

print(
    f"B1 Previous-day WAPE           : "
    f"{b1_wape:.2f}%"
)

print(
    f"B1 improvement vs R0           : "
    f"{b1_improvement:.2f}%"
)

print(
    f"B1 difference vs R1            : "
    f"{b1_wape - r1_stats['Controlled_Injected_WAPE']:+.2f} pp"
)

print("\n" + "-" * 100)
print("PARTIAL-COVERAGE PRACTICAL BASELINES")
print("-" * 100)

print(
    f"B2 Previous-week WAPE          : "
    f"{b2_metrics['WAPE']:.2f}%"
)

print(
    f"B2 row availability            : "
    f"{100 * len(b2_data) / len(baseline_eval):.2f}%"
)

print(
    f"B3 Seasonal-naive WAPE         : "
    f"{b3_metrics['WAPE']:.2f}%"
)

print(
    f"B3 row availability            : "
    f"{100 * len(b3_data) / len(baseline_eval):.2f}%"
)

print("\n" + "-" * 100)
print("EVENT-LEVEL B1 vs R1")
print("-" * 100)

print(
    f"B1 lower WAPE events           : "
    f"{b1_better}/{len(paired)} "
    f"({100*b1_better/len(paired):.2f}%)"
)

print(
    f"Median B1-R1 WAPE difference   : "
    f"{paired['B1_minus_R1_WAPE'].median():+.2f} pp"
)

print(
    f"Wilcoxon p-value               : "
    f"{p_value:.6e}"
)

# ------------------------------------------------------------
# Required output files
# ------------------------------------------------------------

required_outputs = [
    NB6_DATA /
    "B1_PreviousDay_Controlled_July_Predictions.parquet",

    NB6_DATA /
    "B2_PreviousWeek_Controlled_July_Predictions.parquet",

    NB6_DATA /
    "B3_SeasonalNaive_Fallback_Controlled_July_Predictions.parquet",

    NB6_DATA /
    "Event_Level_R0_R1_R2_B1_Comparison.parquet",

    NB6_TABLES /
    "Table_Final_Practical_Baseline_Comparison.csv",

    NB6_TABLES /
    "Table_B2_Matched_Sample_Comparison.csv",

    NB6_TABLES /
    "Table_B3_Matched_Sample_Comparison.csv",

    NB6_TABLES /
    "Table_B1_vs_R1_by_Failure_Duration.csv",

    NB6_FIGURES /
    "Figure_Practical_Baseline_Robustness_WAPE.png",

    NB6_FIGURES /
    "Figure_Robustness_by_Failure_Duration.png",
]

print("\n" + "-" * 100)
print("OUTPUT FILE AUDIT")
print("-" * 100)

all_found = True

for path in required_outputs:

    exists = path.exists()

    if not exists:
        all_found = False

    print(
        f"{'FOUND' if exists else 'MISSING':<8} | "
        f"{path.name}"
    )

# ------------------------------------------------------------
# Save key results as JSON
# ------------------------------------------------------------

json_path = (
    NB6_RESULTS /
    "Notebook_6_Key_Results.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        key_results,
        f,
        indent=4
    )

# Also save as one-row CSV
key_results_csv = (
    NB6_RESULTS /
    "Notebook_6_Key_Results.csv"
)

pd.DataFrame(
    [key_results]
).to_csv(
    key_results_csv,
    index=False
)

# ------------------------------------------------------------
# Final verdict
# ------------------------------------------------------------

print("\n" + "=" * 100)

if all_found:
    print("PASS — NOTEBOOK 6 CORE ANALYSIS COMPLETE.")
else:
    print("CHECK REQUIRED — One or more expected output files are missing.")

print("=" * 100)

print("\nKey-results JSON:")
print(json_path)

print("\nKey-results CSV:")
print(key_results_csv)

In [ ]:
# ============================================================
# CELL 26
# Locate Final August Controlled-Test Artifacts
# ============================================================

import pyarrow.parquet as pq
from pathlib import Path
import pandas as pd

search_roots = [
    NB4_ROOT,
    NB5_ROOT
]

august_candidates = []

for root in search_roots:

    if not root.exists():
        continue

    for path in root.rglob("*.parquet"):

        try:
            pf = pq.ParquetFile(path)
            cols = pf.schema.names

            name_lower = path.name.lower()

            # Keep likely August / controlled / prediction files
            if (
                "august" in name_lower
                or "controlled" in name_lower
                or "injected" in name_lower
            ):

                august_candidates.append({
                    "Notebook": root.name,
                    "File": path.name,
                    "Rows": pf.metadata.num_rows,
                    "Columns": len(cols),
                    "Size_MB": round(
                        path.stat().st_size / (1024**2), 2
                    ),
                    "Path": str(path),
                    "Column_Names": ", ".join(cols)
                })

        except Exception:
            pass

august_candidate_df = pd.DataFrame(
    august_candidates
)

print("=" * 110)
print("AUGUST / CONTROLLED TEST CANDIDATES")
print("=" * 110)

if len(august_candidate_df) == 0:
    print("No candidate files found.")

else:

    august_candidate_df = (
        august_candidate_df
        .sort_values(
            ["Notebook", "File"]
        )
        .reset_index(drop=True)
    )

    for i, row in august_candidate_df.iterrows():

        print("\n" + "-" * 110)

        print(f"[{i}] {row['File']}")
        print(f"Notebook : {row['Notebook']}")
        print(f"Rows     : {row['Rows']:,}")
        print(f"Columns  : {row['Columns']}")
        print(f"Size MB  : {row['Size_MB']}")
        print(f"Path     : {row['Path']}")

        print("\nColumn names:")
        for c in row["Column_Names"].split(", "):
            print(" -", c)


In [ ]:
# ============================================================
# CELL 27
# Load and Audit Final August Controlled Holdout
# ============================================================

august_controlled_path = (
    NB4_ROOT / "Data" /
    "Controlled_Failure_Injected_Affected_Predictions.parquet"
)

august_final_pred_path = (
    NB5_ROOT / "Data" /
    "FINAL_August_Controlled_R0_R1_Predictions.parquet"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

august = pd.read_parquet(
    august_controlled_path
)

august_preds = pd.read_parquet(
    august_final_pred_path
)

august["Timestamp"] = pd.to_datetime(
    august["Timestamp"]
)

august_preds["Timestamp"] = pd.to_datetime(
    august_preds["Timestamp"]
)

# ------------------------------------------------------------
# Derive calendar features needed by R0 model
# ------------------------------------------------------------

august["Interval_Index"] = (
    august["Timestamp"].dt.hour * 4
    + august["Timestamp"].dt.minute // 15
)

august["DayOfWeek"] = (
    august["Timestamp"].dt.dayofweek
)

august["IsWeekend"] = (
    august["DayOfWeek"] >= 5
).astype(int)

# ------------------------------------------------------------
# Basic audit
# ------------------------------------------------------------

print("=" * 90)
print("FINAL AUGUST CONTROLLED HOLDOUT AUDIT")
print("=" * 90)

print(f"Rows               : {len(august):,}")
print(
    f"Events             : "
    f"{august['Injection_Event_ID'].nunique():,}"
)
print(
    f"Detectors          : "
    f"{august['Detector_Code'].nunique():,}"
)

print(
    f"Start timestamp    : "
    f"{august['Timestamp'].min()}"
)

print(
    f"End timestamp      : "
    f"{august['Timestamp'].max()}"
)

print("\nMissing-lag count distribution:")
print(
    august["Missing_Lag_Count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# Check clean/injected lag availability
# ------------------------------------------------------------

lag_names = [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96"
]

print("\nClean lag NaNs:")

for lag in lag_names:

    print(
        f"{lag:<7}: "
        f"{august[lag].isna().sum():,}"
    )

print("\nInjected lag NaNs:")

for lag in lag_names:

    col = f"{lag}_Injected"

    print(
        f"{col:<18}: "
        f"{august[col].isna().sum():,}"
    )

# ------------------------------------------------------------
# Check consistency of target columns
# ------------------------------------------------------------

target_diff = (
    august["Actual"]
    - august["Target_Volume"]
).abs()

print("\nTarget consistency:")
print(
    f"Actual vs Target_Volume max difference : "
    f"{target_diff.max():.12f}"
)

# ------------------------------------------------------------
# Check merge compatibility with final R0/R1 predictions
# ------------------------------------------------------------

keys = [
    "Injection_Event_ID",
    "Detector_Code",
    "Timestamp"
]

merged_check = august[
    keys + ["Actual"]
].merge(
    august_preds,
    on=keys,
    how="left",
    validate="one_to_one",
    suffixes=("_NB4", "_NB5")
)

print("\nFinal prediction file check:")

print(
    f"Matched rows        : "
    f"{merged_check['R0_Injected_Prediction'].notna().sum():,}"
)

print(
    f"R1 matched rows     : "
    f"{merged_check['Final_R1_Injected_Prediction'].notna().sum():,}"
)

print(
    f"Duplicate keys NB4  : "
    f"{august.duplicated(keys).sum():,}"
)

print(
    f"Duplicate keys NB5  : "
    f"{august_preds.duplicated(keys).sum():,}"
)

print("\nCalendar feature check:")
print(
    august[
        [
            "Timestamp",
            "Interval_Index",
            "DayOfWeek",
            "IsWeekend"
        ]
    ]
    .head()
    .to_string(index=False)
)

In [ ]:
# ============================================================
# CELL 28
# Build Historical Lookup for August Practical Baselines
# ============================================================

# ------------------------------------------------------------
# Historical monthly files
# ------------------------------------------------------------

july_history_path = (
    NB3_ROOT / "Data" / "Forecasting_Monthly" /
    "FORECASTING_2026-07.parquet"
)

august_history_path = (
    NB3_ROOT / "Data" / "Forecasting_Monthly" /
    "FORECASTING_2026-08.parquet"
)

print("=" * 90)
print("AUGUST HISTORICAL LOOKUP PREPARATION")
print("=" * 90)

print(f"July file exists   : {july_history_path.exists()}")
print(f"August file exists : {august_history_path.exists()}")

# ------------------------------------------------------------
# Load only required columns
# ------------------------------------------------------------

history_cols = [
    "Timestamp",
    "Detector_Code",
    "Target_Volume"
]

july_hist_aug = pd.read_parquet(
    july_history_path,
    columns=history_cols
)

august_hist = pd.read_parquet(
    august_history_path,
    columns=history_cols
)

# ------------------------------------------------------------
# Combine July + August
# ------------------------------------------------------------

august_history = pd.concat(
    [july_hist_aug, august_hist],
    ignore_index=True
)

august_history["Timestamp"] = pd.to_datetime(
    august_history["Timestamp"]
)

# Keep only the 604 detectors used in August controlled test
august_detectors = set(
    august["Detector_Code"].unique()
)

august_history = august_history[
    august_history["Detector_Code"].isin(
        august_detectors
    )
].copy()

august_history = august_history.sort_values(
    ["Detector_Code", "Timestamp"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

duplicate_keys = august_history.duplicated(
    ["Detector_Code", "Timestamp"]
).sum()

print("\nHistorical lookup audit:")

print(
    f"Rows                 : "
    f"{len(august_history):,}"
)

print(
    f"Detectors            : "
    f"{august_history['Detector_Code'].nunique():,}"
)

print(
    f"Start                : "
    f"{august_history['Timestamp'].min()}"
)

print(
    f"End                  : "
    f"{august_history['Timestamp'].max()}"
)

print(
    f"Duplicate keys       : "
    f"{duplicate_keys:,}"
)

print(
    f"Missing TargetVolume : "
    f"{august_history['Target_Volume'].isna().sum():,}"
)

# ------------------------------------------------------------
# Build lookup:
# (Detector_Code, Timestamp) -> observed volume
# ------------------------------------------------------------

august_history_lookup = (
    august_history
    .set_index(
        ["Detector_Code", "Timestamp"]
    )["Target_Volume"]
)

print("\nLookup entries:")
print(f"{len(august_history_lookup):,}")

print("\nSTATUS:")
if (
    duplicate_keys == 0
    and august_history["Target_Volume"].isna().sum() == 0
    and august_history["Detector_Code"].nunique() == 604
):
    print("PASS — August historical lookup is ready.")
else:
    print("CHECK REQUIRED — Review historical lookup.")

In [ ]:
# ============================================================
# CELL 29
# Build August B1 Previous-Day and B2 Previous-Week Inputs
# ============================================================

august_baseline = august.copy()

lag_steps = {
    "Lag_1": 1,
    "Lag_2": 2,
    "Lag_4": 4,
    "Lag_8": 8,
    "Lag_96": 96
}

# ------------------------------------------------------------
# Build historical substitute values
#
# B1:
# missing Lag_k -> y(t - k - 96)
#
# B2:
# missing Lag_k -> y(t - k - 672)
# ------------------------------------------------------------

for lag_name, lag_k in lag_steps.items():

    prev_day_timestamp = (
        august_baseline["Timestamp"]
        - pd.to_timedelta(
            (lag_k + 96) * 15,
            unit="m"
        )
    )

    prev_week_timestamp = (
        august_baseline["Timestamp"]
        - pd.to_timedelta(
            (lag_k + 672) * 15,
            unit="m"
        )
    )

    day_keys = pd.MultiIndex.from_arrays(
        [
            august_baseline["Detector_Code"],
            prev_day_timestamp
        ],
        names=["Detector_Code", "Timestamp"]
    )

    week_keys = pd.MultiIndex.from_arrays(
        [
            august_baseline["Detector_Code"],
            prev_week_timestamp
        ],
        names=["Detector_Code", "Timestamp"]
    )

    # Historical substitute values
    august_baseline[
        f"{lag_name}_PrevDay"
    ] = (
        august_history_lookup
        .reindex(day_keys)
        .to_numpy()
    )

    august_baseline[
        f"{lag_name}_PrevWeek"
    ] = (
        august_history_lookup
        .reindex(week_keys)
        .to_numpy()
    )

    # --------------------------------------------------------
    # Practical baseline feature values
    # Keep injected lag when available;
    # substitute only when it is missing.
    # --------------------------------------------------------

    august_baseline[
        f"{lag_name}_B1_PrevDay"
    ] = (
        august_baseline[f"{lag_name}_Injected"]
        .fillna(
            august_baseline[f"{lag_name}_PrevDay"]
        )
    )

    august_baseline[
        f"{lag_name}_B2_PrevWeek"
    ] = (
        august_baseline[f"{lag_name}_Injected"]
        .fillna(
            august_baseline[f"{lag_name}_PrevWeek"]
        )
    )


# ------------------------------------------------------------
# Completeness flags
# ------------------------------------------------------------

b1_aug_cols = [
    f"{lag}_B1_PrevDay"
    for lag in lag_steps
]

b2_aug_cols = [
    f"{lag}_B2_PrevWeek"
    for lag in lag_steps
]

august_baseline[
    "B1_All_Lags_Available"
] = (
    august_baseline[b1_aug_cols]
    .notna()
    .all(axis=1)
)

august_baseline[
    "B2_All_Lags_Available"
] = (
    august_baseline[b2_aug_cols]
    .notna()
    .all(axis=1)
)

# ------------------------------------------------------------
# Overall availability
# ------------------------------------------------------------

print("=" * 90)
print("AUGUST PRACTICAL BASELINE INPUT AVAILABILITY")
print("=" * 90)

b1_complete = (
    august_baseline[
        "B1_All_Lags_Available"
    ].sum()
)

b2_complete = (
    august_baseline[
        "B2_All_Lags_Available"
    ].sum()
)

print("\nB1 — Previous-Day Imputation")

print(
    f"Complete rows : "
    f"{b1_complete:,}/{len(august_baseline):,}"
)

print(
    f"Availability  : "
    f"{100*b1_complete/len(august_baseline):.2f}%"
)

print("\nB2 — Previous-Week Imputation")

print(
    f"Complete rows : "
    f"{b2_complete:,}/{len(august_baseline):,}"
)

print(
    f"Availability  : "
    f"{100*b2_complete/len(august_baseline):.2f}%"
)

# ------------------------------------------------------------
# Availability exactly where each injected lag is missing
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("SUBSTITUTE AVAILABILITY WHERE EACH LAG IS ACTUALLY MISSING")
print("=" * 90)

for lag_name in lag_steps:

    needed = (
        august_baseline[
            f"{lag_name}_Injected"
        ].isna()
    )

    n_needed = needed.sum()

    day_available = (
        august_baseline.loc[
            needed,
            f"{lag_name}_PrevDay"
        ]
        .notna()
        .sum()
    )

    week_available = (
        august_baseline.loc[
            needed,
            f"{lag_name}_PrevWeek"
        ]
        .notna()
        .sum()
    )

    print(
        f"{lag_name:<7} | "
        f"Needed: {n_needed:>5,} | "
        f"Prev-Day: {day_available:>5,} "
        f"({100*day_available/n_needed:6.2f}%) | "
        f"Prev-Week: {week_available:>5,} "
        f"({100*week_available/n_needed:6.2f}%)"
    )

# ------------------------------------------------------------
# Event-level full availability
# ------------------------------------------------------------

aug_event_availability = (
    august_baseline
    .groupby("Injection_Event_ID")
    .agg(
        B1_Event_Fully_Available=(
            "B1_All_Lags_Available",
            "all"
        ),
        B2_Event_Fully_Available=(
            "B2_All_Lags_Available",
            "all"
        )
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("EVENT-LEVEL AVAILABILITY")
print("=" * 90)

for col, label in [
    (
        "B1_Event_Fully_Available",
        "B1 Previous-Day"
    ),
    (
        "B2_Event_Fully_Available",
        "B2 Previous-Week"
    )
]:

    n = aug_event_availability[col].sum()
    total = len(aug_event_availability)

    print(
        f"{label:<20}: "
        f"{n:,}/{total:,} "
        f"({100*n/total:.2f}%)"
    )

In [ ]:
# ============================================================
# CELL 30
# Diagnose Missing Previous-Week Substitutions in August
# ============================================================

print("=" * 90)
print("B2 PREVIOUS-WEEK MISSING-HISTORY DIAGNOSTIC")
print("=" * 90)

diagnostic_rows = []

for lag_name in lag_steps:

    needed = (
        august_baseline[f"{lag_name}_Injected"].isna()
    )

    missing_week = (
        august_baseline[f"{lag_name}_PrevWeek"].isna()
    )

    problem = august_baseline[
        needed & missing_week
    ].copy()

    print(
        f"\n{lag_name}: "
        f"{len(problem):,} required substitutions unavailable"
    )

    if len(problem) > 0:

        print(
            f"Unique events    : "
            f"{problem['Injection_Event_ID'].nunique():,}"
        )

        print(
            f"Unique detectors : "
            f"{problem['Detector_Code'].nunique():,}"
        )

        print(
            f"Timestamp range  : "
            f"{problem['Timestamp'].min()} "
            f"to {problem['Timestamp'].max()}"
        )

        for _, row in problem.iterrows():

            lag_k = lag_steps[lag_name]

            required_timestamp = (
                row["Timestamp"]
                - pd.to_timedelta(
                    (lag_k + 672) * 15,
                    unit="m"
                )
            )

            diagnostic_rows.append({
                "Lag": lag_name,
                "Injection_Event_ID": row["Injection_Event_ID"],
                "Detector_Code": row["Detector_Code"],
                "Forecast_Timestamp": row["Timestamp"],
                "Required_PreviousWeek_Timestamp": required_timestamp
            })


# ------------------------------------------------------------
# Consolidate diagnostic table
# ------------------------------------------------------------

b2_missing_diag = pd.DataFrame(
    diagnostic_rows
)

print("\n" + "=" * 90)
print("OVERALL DIAGNOSTIC")
print("=" * 90)

print(
    f"Unique affected events    : "
    f"{b2_missing_diag['Injection_Event_ID'].nunique():,}"
)

print(
    f"Unique affected detectors : "
    f"{b2_missing_diag['Detector_Code'].nunique():,}"
)

print(
    f"Required history start    : "
    f"{b2_missing_diag['Required_PreviousWeek_Timestamp'].min()}"
)

print(
    f"Required history end      : "
    f"{b2_missing_diag['Required_PreviousWeek_Timestamp'].max()}"
)

print("\nAffected detectors:")
print(
    sorted(
        b2_missing_diag["Detector_Code"]
        .unique()
        .tolist()
    )
)

print("\nSample missing-history rows:")
display(
    b2_missing_diag.head(30)
)


# ------------------------------------------------------------
# Check whether affected detectors exist in July history
# ------------------------------------------------------------

july_detector_set = set(
    july_hist_aug["Detector_Code"].unique()
)

affected_detectors = set(
    b2_missing_diag["Detector_Code"].unique()
)

missing_from_july = sorted(
    affected_detectors - july_detector_set
)

print("\n" + "=" * 90)
print("JULY DETECTOR PRESENCE CHECK")
print("=" * 90)

print(
    f"Affected detectors absent from July file: "
    f"{len(missing_from_july):,}"
)

print(missing_from_july)

In [ ]:
# ============================================================
# CELL 31
# Verify Original R0 Model Against Final August Predictions
# ============================================================

# ------------------------------------------------------------
# Build clean August feature matrix
# ------------------------------------------------------------

X_aug_clean = pd.DataFrame({
    "Detector_Code": august_baseline["Detector_Code"],
    "Interval_Index": august_baseline["Interval_Index"],
    "DayOfWeek": august_baseline["DayOfWeek"],
    "IsWeekend": august_baseline["IsWeekend"],
    "Lag_1": august_baseline["Lag_1"],
    "Lag_2": august_baseline["Lag_2"],
    "Lag_4": august_baseline["Lag_4"],
    "Lag_8": august_baseline["Lag_8"],
    "Lag_96": august_baseline["Lag_96"],
})

# ------------------------------------------------------------
# Predict with loaded R0 model
# ------------------------------------------------------------

aug_r0_raw = r0_model.predict(X_aug_clean)

# Same post-processing used previously
aug_r0_reproduced = np.maximum(
    aug_r0_raw,
    0
)

# ------------------------------------------------------------
# Merge stored final August R0 predictions
# ------------------------------------------------------------

aug_verify = august_baseline[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ]
].copy()

aug_verify["R0_Reproduced"] = (
    aug_r0_reproduced
)

aug_verify = aug_verify.merge(
    august_preds[
        [
            "Injection_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Clean_Prediction"
        ]
    ],
    on=[
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Difference
# ------------------------------------------------------------

aug_verify[
    "Absolute_Difference"
] = (
    aug_verify["R0_Reproduced"]
    - aug_verify["R0_Clean_Prediction"]
).abs()

print("=" * 90)
print("AUGUST R0 REPRODUCIBILITY CHECK")
print("=" * 90)

print(
    f"Rows checked             : "
    f"{len(aug_verify):,}"
)

print(
    f"Stored predictions found : "
    f"{aug_verify['R0_Clean_Prediction'].notna().sum():,}"
)

print(
    f"Maximum abs difference   : "
    f"{aug_verify['Absolute_Difference'].max():.12f}"
)

print(
    f"Mean abs difference      : "
    f"{aug_verify['Absolute_Difference'].mean():.12f}"
)

print(
    f"Rows difference > 1e-10  : "
    f"{(aug_verify['Absolute_Difference'] > 1e-10).sum():,}"
)

print(
    f"Rows difference > 1e-8   : "
    f"{(aug_verify['Absolute_Difference'] > 1e-8).sum():,}"
)

print(
    f"Raw negative predictions : "
    f"{(aug_r0_raw < 0).sum():,}"
)

# ------------------------------------------------------------
# Verdict
# ------------------------------------------------------------

max_diff = (
    aug_verify[
        "Absolute_Difference"
    ].max()
)

print("\n" + "=" * 90)

if max_diff < 1e-8:
    print(
        "PASS — Loaded R0 exactly reproduces "
        "the final August R0 predictions."
    )
else:
    print(
        "CHECK REQUIRED — Final August R0 "
        "is not reproduced exactly."
    )

print("=" * 90)

In [ ]:
# ============================================================
# CELL 32
# Locate Final R0 / August Model Artifacts in Notebook 5
# ============================================================

from pathlib import Path
import os

print("=" * 100)
print("NOTEBOOK 5 MODEL ARTIFACT SEARCH")
print("=" * 100)

model_extensions = {
    ".txt",
    ".json",
    ".ubj",
    ".bin",
    ".pkl",
    ".pickle",
    ".joblib"
}

model_candidates = []

for path in NB5_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() in model_extensions:

        name_lower = path.name.lower()

        model_candidates.append({
            "File": path.name,
            "Size_MB": round(
                path.stat().st_size / (1024**2),
                3
            ),
            "Path": str(path),
            "Likely_R0": (
                "r0" in name_lower
                or "clean" in name_lower
            ),
            "Likely_R1": (
                "r1" in name_lower
                or "failureaware" in name_lower
                or "failure_aware" in name_lower
            ),
            "Likely_Final": (
                "final" in name_lower
                or "janjul" in name_lower
                or "august" in name_lower
            )
        })

model_candidates_df = pd.DataFrame(
    model_candidates
)

if len(model_candidates_df) == 0:

    print("No model files found in Notebook 5.")

else:

    model_candidates_df = (
        model_candidates_df
        .sort_values(
            [
                "Likely_Final",
                "Likely_R0",
                "Likely_R1",
                "File"
            ],
            ascending=[
                False,
                False,
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )

    print(
        f"Model files found: "
        f"{len(model_candidates_df)}\n"
    )

    for i, row in model_candidates_df.iterrows():

        print("-" * 100)

        print(f"[{i}] {row['File']}")
        print(f"Size MB      : {row['Size_MB']}")
        print(f"Likely R0    : {row['Likely_R0']}")
        print(f"Likely R1    : {row['Likely_R1']}")
        print(f"Likely Final : {row['Likely_Final']}")
        print(f"Path         : {row['Path']}")

In [ ]:
# ============================================================
# CELL 33
# Inspect R0 Original Model Recovery Metadata
# ============================================================

import json
from pathlib import Path

r0_recovery_path = (
    NB5_ROOT / "Results" /
    "R0_Original_Model_Recovery.json"
)

recovery_summary_path = (
    NB5_ROOT / "Results" /
    "NOTEBOOK_5_FINAL_RECOVERY_SUMMARY.txt"
)

print("=" * 100)
print("R0 ORIGINAL MODEL RECOVERY METADATA")
print("=" * 100)

print(f"\nJSON path:")
print(r0_recovery_path)
print(f"Exists: {r0_recovery_path.exists()}")

if r0_recovery_path.exists():

    with open(
        r0_recovery_path,
        "r",
        encoding="utf-8"
    ) as f:
        r0_recovery = json.load(f)

    print("\nJSON CONTENT:")
    print(
        json.dumps(
            r0_recovery,
            indent=4
        )
    )

else:
    print("R0 recovery JSON not found.")


print("\n" + "=" * 100)
print("NOTEBOOK 5 FINAL RECOVERY SUMMARY")
print("=" * 100)

print(f"\nTXT path:")
print(recovery_summary_path)
print(f"Exists: {recovery_summary_path.exists()}")

if recovery_summary_path.exists():

    with open(
        recovery_summary_path,
        "r",
        encoding="utf-8"
    ) as f:
        recovery_text = f.read()

    print("\nTEXT CONTENT:")
    print(recovery_text)

else:
    print("Recovery summary TXT not found.")

In [ ]:
# ============================================================
# CELL 34 — CORRECTED
# Verify Original August Features and Reproduce R0
# ============================================================

# ------------------------------------------------------------
# Load original August forecasting features
# ------------------------------------------------------------

aug_source_cols = [
    "Timestamp",
    "Detector_Code",
    "Interval_Index",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96",
    "Target_Volume"
]

aug_source = pd.read_parquet(
    august_history_path,
    columns=aug_source_cols
)

aug_source["Timestamp"] = pd.to_datetime(
    aug_source["Timestamp"]
)

# ------------------------------------------------------------
# Keep controlled clean features too, so suffixes are created
# ------------------------------------------------------------

controlled_feature_cols = [
    "Injection_Event_ID",
    "Detector_Code",
    "Timestamp",
    "Interval_Index",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96",
    "Target_Volume"
]

feature_check = august_baseline[
    controlled_feature_cols
].merge(
    aug_source,
    on=[
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="many_to_one",
    suffixes=(
        "_Controlled",
        "_Original"
    )
)

print("=" * 95)
print("ORIGINAL vs CONTROLLED AUGUST FEATURE CODING")
print("=" * 95)

print(
    f"Rows matched: "
    f"{feature_check['Target_Volume_Original'].notna().sum():,}"
    f"/{len(feature_check):,}"
)

# ------------------------------------------------------------
# Compare calendar features
# ------------------------------------------------------------

for col in [
    "Interval_Index",
    "DayOfWeek",
    "IsWeekend"
]:

    diff = (
        feature_check[f"{col}_Controlled"]
        != feature_check[f"{col}_Original"]
    ).sum()

    print(
        f"{col:<15} differences: {diff:,}"
    )

# ------------------------------------------------------------
# Compare target
# ------------------------------------------------------------

target_diff = np.abs(
    feature_check["Target_Volume_Controlled"]
    - feature_check["Target_Volume_Original"]
)

print(
    f"\nTarget max difference : "
    f"{target_diff.max():.12f}"
)

print(
    f"Target rows > 1e-10   : "
    f"{(target_diff > 1e-10).sum():,}"
)

# ------------------------------------------------------------
# Compare clean lag values
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("CLEAN LAG CONSISTENCY")
print("=" * 95)

for lag in [
    "Lag_1",
    "Lag_2",
    "Lag_4",
    "Lag_8",
    "Lag_96"
]:

    original = feature_check[
        f"{lag}_Original"
    ].to_numpy()

    controlled = feature_check[
        f"{lag}_Controlled"
    ].to_numpy()

    diff = np.abs(
        original - controlled
    )

    print(
        f"{lag:<7} | "
        f"Max difference: {np.nanmax(diff):.12f} | "
        f"Rows > 1e-10: {(diff > 1e-10).sum():,}"
    )

# ------------------------------------------------------------
# Reproduce R0 using ORIGINAL monthly August features
# ------------------------------------------------------------

X_aug_original = pd.DataFrame({
    "Detector_Code":
        feature_check["Detector_Code"],

    "Interval_Index":
        feature_check["Interval_Index_Original"],

    "DayOfWeek":
        feature_check["DayOfWeek_Original"],

    "IsWeekend":
        feature_check["IsWeekend_Original"],

    "Lag_1":
        feature_check["Lag_1_Original"],

    "Lag_2":
        feature_check["Lag_2_Original"],

    "Lag_4":
        feature_check["Lag_4_Original"],

    "Lag_8":
        feature_check["Lag_8_Original"],

    "Lag_96":
        feature_check["Lag_96_Original"]
})

r0_aug_original_raw = r0_model.predict(
    X_aug_original
)

# Same post-processing
r0_aug_original = np.maximum(
    r0_aug_original_raw,
    0
)

# ------------------------------------------------------------
# Compare against stored final August R0 clean predictions
# ------------------------------------------------------------

repro_check = feature_check[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ]
].copy()

repro_check[
    "R0_Reproduced"
] = r0_aug_original

repro_check = repro_check.merge(
    august_preds[
        [
            "Injection_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Clean_Prediction"
        ]
    ],
    on=[
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

repro_check[
    "Absolute_Difference"
] = (
    repro_check["R0_Reproduced"]
    - repro_check["R0_Clean_Prediction"]
).abs()

print("\n" + "=" * 95)
print("R0 REPRODUCTION USING ORIGINAL AUGUST FEATURES")
print("=" * 95)

print(
    f"Maximum abs difference : "
    f"{repro_check['Absolute_Difference'].max():.12f}"
)

print(
    f"Mean abs difference    : "
    f"{repro_check['Absolute_Difference'].mean():.12f}"
)

print(
    f"Rows > 1e-6           : "
    f"{(repro_check['Absolute_Difference'] > 1e-6).sum():,}"
)

print(
    f"Rows > 1e-4           : "
    f"{(repro_check['Absolute_Difference'] > 1e-4).sum():,}"
)

print(
    f"Raw negative predictions clipped: "
    f"{(r0_aug_original_raw < 0).sum():,}"
)

# ------------------------------------------------------------
# Verdict
# ------------------------------------------------------------

max_diff = (
    repro_check[
        "Absolute_Difference"
    ].max()
)

print("\n" + "=" * 95)

if max_diff < 1e-4:

    print(
        "PASS — R0 August predictions reproduced "
        "within the documented numerical tolerance."
    )

else:

    print(
        "CHECK REQUIRED — A substantive discrepancy remains."
    )

print("=" * 95)

In [ ]:
# ============================================================
# CELL 35
# Locate Original August Prediction Artifacts from Notebook 3
# ============================================================

import pyarrow.parquet as pq
import pandas as pd

print("=" * 105)
print("SEARCH NOTEBOOK 3 FOR AUGUST PREDICTION FILES")
print("=" * 105)

aug_prediction_candidates = []

for path in NB3_ROOT.rglob("*.parquet"):

    try:
        pf = pq.ParquetFile(path)
        cols = pf.schema.names

        name_lower = path.name.lower()

        prediction_cols = [
            c for c in cols
            if (
                "predict" in c.lower()
                or "lightgbm" in c.lower()
                or "xgboost" in c.lower()
                or "lgb" in c.lower()
                or "xgb" in c.lower()
            )
        ]

        if (
            "august" in name_lower
            or "2026-08" in name_lower
            or len(prediction_cols) > 0
        ):

            aug_prediction_candidates.append({
                "File": path.name,
                "Rows": pf.metadata.num_rows,
                "Columns": len(cols),
                "Prediction_Columns": prediction_cols,
                "Path": str(path)
            })

    except Exception:
        pass


aug_pred_df = pd.DataFrame(
    aug_prediction_candidates
)

if len(aug_pred_df) == 0:

    print("No relevant files found.")

else:

    aug_pred_df = (
        aug_pred_df
        .sort_values(
            ["File"]
        )
        .reset_index(drop=True)
    )

    print(
        f"Candidate files found: "
        f"{len(aug_pred_df)}\n"
    )

    for i, row in aug_pred_df.iterrows():

        print("-" * 105)

        print(f"[{i}] {row['File']}")
        print(f"Rows   : {row['Rows']:,}")
        print(f"Columns: {row['Columns']}")
        print(f"Path   : {row['Path']}")

        print("\nPrediction-related columns:")

        if len(row["Prediction_Columns"]) == 0:
            print(" - None")
        else:
            for c in row["Prediction_Columns"]:
                print(" -", c)

In [ ]:
# ============================================================
# CELL 36
# Trace the Exact Source of Final August R0 Predictions
# ============================================================

aug_test_pred_path = (
    NB3_ROOT / "Data" /
    "August_Final_Test_Predictions.parquet"
)

aug_all_pred_path = (
    NB3_ROOT / "Data" /
    "August_Final_All_Model_Predictions.parquet"
)

print("=" * 100)
print("TRACE FINAL AUGUST R0 PREDICTION SOURCE")
print("=" * 100)

# ------------------------------------------------------------
# Inspect schemas
# ------------------------------------------------------------

for label, path in [
    ("August Final Test Predictions", aug_test_pred_path),
    ("August Final All-Model Predictions", aug_all_pred_path)
]:

    pf = pq.ParquetFile(path)

    print("\n" + "-" * 100)
    print(label)
    print("-" * 100)

    print(f"Rows : {pf.metadata.num_rows:,}")
    print("Columns:")

    for c in pf.schema.names:
        print(" -", c)


# ------------------------------------------------------------
# Load both files
# ------------------------------------------------------------

aug_test_pred = pd.read_parquet(
    aug_test_pred_path
)

aug_all_pred = pd.read_parquet(
    aug_all_pred_path
)

for df in [aug_test_pred, aug_all_pred]:
    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"]
    )

# ------------------------------------------------------------
# Compare Notebook 3 prediction files to each other
# ------------------------------------------------------------

common_keys = [
    "Detector_Code",
    "Timestamp"
]

test_vs_all = aug_test_pred[
    common_keys + ["LightGBM_Prediction"]
].merge(
    aug_all_pred[
        common_keys + ["LightGBM_Prediction"]
    ],
    on=common_keys,
    how="inner",
    validate="one_to_one",
    suffixes=("_Test", "_All")
)

test_vs_all["Diff"] = (
    test_vs_all["LightGBM_Prediction_Test"]
    - test_vs_all["LightGBM_Prediction_All"]
).abs()

print("\n" + "=" * 100)
print("NOTEBOOK 3 INTERNAL CONSISTENCY")
print("=" * 100)

print(
    f"Matched rows              : "
    f"{len(test_vs_all):,}"
)

print(
    f"Max LightGBM difference   : "
    f"{test_vs_all['Diff'].max():.12f}"
)

print(
    f"Mean LightGBM difference  : "
    f"{test_vs_all['Diff'].mean():.12f}"
)

# ------------------------------------------------------------
# Compare Notebook 3 August predictions to Notebook 5 R0
# on the SAME 12,662 controlled rows
# ------------------------------------------------------------

source_compare = august_preds[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "R0_Clean_Prediction"
    ]
].merge(
    aug_test_pred[
        common_keys + ["LightGBM_Prediction"]
    ],
    on=common_keys,
    how="left",
    validate="many_to_one"
)

source_compare[
    "Notebook3_vs_Notebook5_Diff"
] = (
    source_compare["LightGBM_Prediction"]
    - source_compare["R0_Clean_Prediction"]
).abs()

print("\n" + "=" * 100)
print("NOTEBOOK 3 vs NOTEBOOK 5 R0")
print("=" * 100)

print(
    f"Rows matched              : "
    f"{source_compare['LightGBM_Prediction'].notna().sum():,}"
)

print(
    f"Max absolute difference   : "
    f"{source_compare['Notebook3_vs_Notebook5_Diff'].max():.12f}"
)

print(
    f"Mean absolute difference  : "
    f"{source_compare['Notebook3_vs_Notebook5_Diff'].mean():.12f}"
)

print(
    f"Rows difference > 1e-6    : "
    f"{(source_compare['Notebook3_vs_Notebook5_Diff'] > 1e-6).sum():,}"
)

print(
    f"Rows difference > 1e-4    : "
    f"{(source_compare['Notebook3_vs_Notebook5_Diff'] > 1e-4).sum():,}"
)

# ------------------------------------------------------------
# Compare current model reproduction to Notebook 3 predictions
# ------------------------------------------------------------

model_vs_nb3 = repro_check[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "R0_Reproduced"
    ]
].merge(
    aug_test_pred[
        common_keys + ["LightGBM_Prediction"]
    ],
    on=common_keys,
    how="left",
    validate="many_to_one"
)

model_vs_nb3[
    "Model_vs_Notebook3_Diff"
] = (
    model_vs_nb3["R0_Reproduced"]
    - model_vs_nb3["LightGBM_Prediction"]
).abs()

print("\n" + "=" * 100)
print("CURRENT MODEL FILE vs NOTEBOOK 3 STORED AUGUST PREDICTIONS")
print("=" * 100)

print(
    f"Max absolute difference   : "
    f"{model_vs_nb3['Model_vs_Notebook3_Diff'].max():.12f}"
)

print(
    f"Mean absolute difference  : "
    f"{model_vs_nb3['Model_vs_Notebook3_Diff'].mean():.12f}"
)

print(
    f"Rows difference > 1e-4    : "
    f"{(model_vs_nb3['Model_vs_Notebook3_Diff'] > 1e-4).sum():,}"
)

# ------------------------------------------------------------
# Verdict
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SOURCE VERDICT")
print("=" * 100)

nb3_nb5_max = (
    source_compare[
        "Notebook3_vs_Notebook5_Diff"
    ].max()
)

if nb3_nb5_max < 1e-4:

    print(
        "PASS — Notebook 5 R0 clean predictions came from "
        "the stored Notebook 3 August prediction output."
    )

else:

    print(
        "CHECK REQUIRED — Notebook 5 R0 clean predictions "
        "do not match the stored Notebook 3 August predictions."
    )

In [ ]:
# ============================================================
# CELL 37
# Identify the Exact LightGBM Model Behind Stored August Predictions
# ============================================================

import lightgbm as lgb
from pathlib import Path

print("=" * 105)
print("SEARCH FOR EXACT LIGHTGBM MODEL MATCH")
print("=" * 105)

# ------------------------------------------------------------
# Find all likely LightGBM model files in Notebook 3
# ------------------------------------------------------------

candidate_model_paths = []

for path in NB3_ROOT.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        path.suffix.lower() in [".txt", ".model", ".bin"]
        and (
            "lightgbm" in name
            or "lgb" in name
        )
    ):
        candidate_model_paths.append(path)

print(f"Candidate LightGBM model files found: {len(candidate_model_paths)}\n")

for i, path in enumerate(candidate_model_paths):
    print(f"[{i}] {path}")

# ------------------------------------------------------------
# Use SAME controlled August clean feature matrix
# ------------------------------------------------------------

X_model_test = pd.DataFrame({
    "Detector_Code": feature_check["Detector_Code"],
    "Interval_Index": feature_check["Interval_Index_Original"],
    "DayOfWeek": feature_check["DayOfWeek_Original"],
    "IsWeekend": feature_check["IsWeekend_Original"],
    "Lag_1": feature_check["Lag_1_Original"],
    "Lag_2": feature_check["Lag_2_Original"],
    "Lag_4": feature_check["Lag_4_Original"],
    "Lag_8": feature_check["Lag_8_Original"],
    "Lag_96": feature_check["Lag_96_Original"]
})

# Stored Notebook 3 predictions on same 12,662 rows
stored_aug = source_compare[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "LightGBM_Prediction"
    ]
].copy()

# ------------------------------------------------------------
# Test every candidate model
# ------------------------------------------------------------

results = []

for path in candidate_model_paths:

    try:
        model = lgb.Booster(
            model_file=str(path)
        )

        raw_pred = model.predict(
            X_model_test
        )

        pred = np.maximum(
            raw_pred,
            0
        )

        test_df = feature_check[
            [
                "Injection_Event_ID",
                "Detector_Code",
                "Timestamp"
            ]
        ].copy()

        test_df["Candidate_Prediction"] = pred

        test_df = test_df.merge(
            stored_aug,
            on=[
                "Injection_Event_ID",
                "Detector_Code",
                "Timestamp"
            ],
            how="left",
            validate="one_to_one"
        )

        diff = np.abs(
            test_df["Candidate_Prediction"]
            - test_df["LightGBM_Prediction"]
        )

        results.append({
            "Model_File": path.name,
            "Path": str(path),
            "Trees": model.num_trees(),
            "Features": len(model.feature_name()),
            "Max_Abs_Diff": diff.max(),
            "Mean_Abs_Diff": diff.mean(),
            "Rows_GT_1e_6": int((diff > 1e-6).sum()),
            "Rows_GT_1e_4": int((diff > 1e-4).sum()),
            "Raw_Negative_Predictions": int((raw_pred < 0).sum())
        })

    except Exception as e:

        results.append({
            "Model_File": path.name,
            "Path": str(path),
            "Trees": np.nan,
            "Features": np.nan,
            "Max_Abs_Diff": np.nan,
            "Mean_Abs_Diff": np.nan,
            "Rows_GT_1e_6": np.nan,
            "Rows_GT_1e_4": np.nan,
            "Raw_Negative_Predictions": np.nan,
            "Error": str(e)
        })

# ------------------------------------------------------------
# Display ranked results
# ------------------------------------------------------------

model_match_df = pd.DataFrame(results)

if len(model_match_df) > 0:

    model_match_df = model_match_df.sort_values(
        [
            "Rows_GT_1e_4",
            "Mean_Abs_Diff"
        ],
        ascending=True
    ).reset_index(drop=True)

    print("\n" + "=" * 105)
    print("MODEL MATCH RESULTS")
    print("=" * 105)

    display(model_match_df)

    best = model_match_df.iloc[0]

    print("\n" + "=" * 105)
    print("BEST MATCH")
    print("=" * 105)

    print(f"Model file       : {best['Model_File']}")
    print(f"Path             : {best['Path']}")
    print(f"Trees            : {best['Trees']}")
    print(f"Features         : {best['Features']}")
    print(f"Max abs diff     : {best['Max_Abs_Diff']:.12f}")
    print(f"Mean abs diff    : {best['Mean_Abs_Diff']:.12f}")
    print(f"Rows > 1e-4      : {best['Rows_GT_1e_4']}")

    if best["Max_Abs_Diff"] < 1e-4:
        print("\nPASS — Exact historical R0 model identified.")
    else:
        print("\nCHECK REQUIRED — No candidate exactly reproduces stored August predictions.")

else:

    print("No candidate LightGBM model files found.")

In [ ]:
# ============================================================
# CELL 38
# Final August Evaluation:
# B1 Previous-Day and B2 Previous-Week Imputation
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb

# ------------------------------------------------------------
# Load the exact historical R0 model identified in Cell 37
# ------------------------------------------------------------

final_r0_model_path = (
    NB3_ROOT / "Models" /
    "LightGBM_FINAL_JanJul_1200.txt"
)

final_r0_model = lgb.Booster(
    model_file=str(final_r0_model_path)
)

print("=" * 95)
print("FINAL AUGUST PRACTICAL BASELINE EVALUATION")
print("=" * 95)

print(f"R0 model:")
print(final_r0_model_path)

# ------------------------------------------------------------
# Metric helper
# ------------------------------------------------------------

def august_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    wape = (
        np.abs(y_true - y_pred).sum()
        / np.abs(y_true).sum()
        * 100
    )

    bias = np.mean(
        y_pred - y_true
    )

    return {
        "WAPE": wape,
        "MAE": mae,
        "RMSE": rmse,
        "Bias": bias
    }


# ============================================================
# B1 — PREVIOUS-DAY
# ============================================================

X_aug_b1 = pd.DataFrame({
    "Detector_Code":
        august_baseline["Detector_Code"],

    "Interval_Index":
        august_baseline["Interval_Index"],

    "DayOfWeek":
        august_baseline["DayOfWeek"],

    "IsWeekend":
        august_baseline["IsWeekend"],

    "Lag_1":
        august_baseline["Lag_1_B1_PrevDay"],

    "Lag_2":
        august_baseline["Lag_2_B1_PrevDay"],

    "Lag_4":
        august_baseline["Lag_4_B1_PrevDay"],

    "Lag_8":
        august_baseline["Lag_8_B1_PrevDay"],

    "Lag_96":
        august_baseline["Lag_96_B1_PrevDay"]
})

assert X_aug_b1.isna().sum().sum() == 0

b1_aug_raw = final_r0_model.predict(
    X_aug_b1
)

b1_aug_pred = np.maximum(
    b1_aug_raw,
    0
)

actual_aug = (
    august_baseline["Target_Volume"]
    .to_numpy()
)

b1_aug_metrics = august_metrics(
    actual_aug,
    b1_aug_pred
)


# ============================================================
# STORED FINAL R0 AND R1 — FULL AUGUST SET
# ============================================================

aug_reference = august_baseline[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume"
    ]
].merge(
    august_preds[
        [
            "Injection_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Injected_Prediction",
            "Final_R1_Injected_Prediction"
        ]
    ],
    on=[
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

r0_aug_metrics = august_metrics(
    aug_reference["Target_Volume"],
    aug_reference["R0_Injected_Prediction"]
)

r1_aug_metrics = august_metrics(
    aug_reference["Target_Volume"],
    aug_reference["Final_R1_Injected_Prediction"]
)


# ============================================================
# B2 — PREVIOUS-WEEK
# ============================================================

b2_aug_mask = (
    august_baseline[
        "B2_All_Lags_Available"
    ]
)

b2_aug_data = (
    august_baseline.loc[
        b2_aug_mask
    ].copy()
)

X_aug_b2 = pd.DataFrame({
    "Detector_Code":
        b2_aug_data["Detector_Code"],

    "Interval_Index":
        b2_aug_data["Interval_Index"],

    "DayOfWeek":
        b2_aug_data["DayOfWeek"],

    "IsWeekend":
        b2_aug_data["IsWeekend"],

    "Lag_1":
        b2_aug_data["Lag_1_B2_PrevWeek"],

    "Lag_2":
        b2_aug_data["Lag_2_B2_PrevWeek"],

    "Lag_4":
        b2_aug_data["Lag_4_B2_PrevWeek"],

    "Lag_8":
        b2_aug_data["Lag_8_B2_PrevWeek"],

    "Lag_96":
        b2_aug_data["Lag_96_B2_PrevWeek"]
})

assert X_aug_b2.isna().sum().sum() == 0

b2_aug_raw = final_r0_model.predict(
    X_aug_b2
)

b2_aug_pred = np.maximum(
    b2_aug_raw,
    0
)

b2_aug_metrics = august_metrics(
    b2_aug_data["Target_Volume"],
    b2_aug_pred
)

# ------------------------------------------------------------
# Matched R0/R1 comparison for exactly same B2 rows
# ------------------------------------------------------------

b2_matched_aug = b2_aug_data[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume"
    ]
].copy()

b2_matched_aug["B2_Prediction"] = (
    b2_aug_pred
)

b2_matched_aug = b2_matched_aug.merge(
    august_preds[
        [
            "Injection_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Injected_Prediction",
            "Final_R1_Injected_Prediction"
        ]
    ],
    on=[
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

r0_b2_matched_metrics = august_metrics(
    b2_matched_aug["Target_Volume"],
    b2_matched_aug["R0_Injected_Prediction"]
)

r1_b2_matched_metrics = august_metrics(
    b2_matched_aug["Target_Volume"],
    b2_matched_aug["Final_R1_Injected_Prediction"]
)


# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 95)
print("FULL AUGUST CONTROLLED SET — 12,662 ROWS")
print("=" * 95)

print(
    f"R0 WAPE : {r0_aug_metrics['WAPE']:.6f}%"
)
print(
    f"R1 WAPE : {r1_aug_metrics['WAPE']:.6f}%"
)
print(
    f"B1 WAPE : {b1_aug_metrics['WAPE']:.6f}%"
)

print("\nB1:")
print(
    f"Availability : 100.00%"
)
print(
    f"MAE          : {b1_aug_metrics['MAE']:.6f}"
)
print(
    f"RMSE         : {b1_aug_metrics['RMSE']:.6f}"
)
print(
    f"Bias         : {b1_aug_metrics['Bias']:.6f}"
)

b1_vs_r0_improvement = (
    (
        r0_aug_metrics["WAPE"]
        - b1_aug_metrics["WAPE"]
    )
    / r0_aug_metrics["WAPE"]
    * 100
)

print(
    f"Improvement vs R0 : "
    f"{b1_vs_r0_improvement:.3f}%"
)

print(
    f"B1 minus R1 WAPE  : "
    f"{b1_aug_metrics['WAPE'] - r1_aug_metrics['WAPE']:+.4f} pp"
)


print("\n" + "=" * 95)
print(
    f"B2 AVAILABLE AUGUST ROWS — "
    f"{len(b2_aug_data):,}/{len(august_baseline):,}"
)
print("=" * 95)

print(
    f"Availability : "
    f"{100*len(b2_aug_data)/len(august_baseline):.2f}%"
)

print(
    f"B2 WAPE      : "
    f"{b2_aug_metrics['WAPE']:.6f}%"
)

print(
    f"B2 MAE       : "
    f"{b2_aug_metrics['MAE']:.6f}"
)

print(
    f"B2 RMSE      : "
    f"{b2_aug_metrics['RMSE']:.6f}"
)

print(
    f"B2 Bias      : "
    f"{b2_aug_metrics['Bias']:.6f}"
)

print("\nMatched R0/R1 on SAME B2 rows:")

print(
    f"R0 matched WAPE : "
    f"{r0_b2_matched_metrics['WAPE']:.6f}%"
)

print(
    f"R1 matched WAPE : "
    f"{r1_b2_matched_metrics['WAPE']:.6f}%"
)

print(
    f"B2 minus R1     : "
    f"{b2_aug_metrics['WAPE'] - r1_b2_matched_metrics['WAPE']:+.4f} pp"
)


# ============================================================
# Save predictions
# ============================================================

aug_practical_output = august_baseline[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume",
        "Missing_Lag_Count",
        "Missing_Lag_Pattern"
    ]
].copy()

aug_practical_output[
    "B1_PreviousDay_Prediction"
] = b1_aug_pred

aug_practical_output[
    "B2_PreviousWeek_Prediction"
] = np.nan

aug_practical_output.loc[
    b2_aug_mask,
    "B2_PreviousWeek_Prediction"
] = b2_aug_pred

aug_practical_output[
    "B2_Available"
] = b2_aug_mask.to_numpy()

aug_practical_output_path = (
    NB6_DATA /
    "FINAL_August_Practical_Baseline_Predictions.parquet"
)

aug_practical_output.to_parquet(
    aug_practical_output_path,
    index=False
)

print("\nSaved:")
print(aug_practical_output_path)

In [ ]:
# ============================================================
# CELL 39
# Final August Evaluation:
# B3 Seasonal-Naive Fallback
# ============================================================

print("=" * 95)
print("FINAL AUGUST B3 — SEASONAL-NAIVE FALLBACK")
print("=" * 95)

# ------------------------------------------------------------
# B3 definition:
# forecast at t = observation at t-96 (24 h earlier)
#
# If Lag_96 is unavailable after failure injection,
# B3 is considered unavailable.
# ------------------------------------------------------------

b3_aug_mask = (
    august_baseline["Lag_96_Injected"].notna()
)

b3_aug_data = (
    august_baseline.loc[
        b3_aug_mask
    ].copy()
)

b3_aug_pred = (
    b3_aug_data["Lag_96_Injected"]
    .to_numpy()
)

# ------------------------------------------------------------
# Verify against stored Notebook 4 seasonal-naive prediction
# ------------------------------------------------------------

stored_seasonal = (
    b3_aug_data[
        "Seasonal_Naive_Injected_Prediction"
    ].to_numpy()
)

seasonal_diff = np.abs(
    b3_aug_pred - stored_seasonal
)

print(
    f"Available rows : "
    f"{len(b3_aug_data):,}/{len(august_baseline):,}"
)

print(
    f"Availability   : "
    f"{100*len(b3_aug_data)/len(august_baseline):.2f}%"
)

print("\nStored seasonal-naive consistency:")

print(
    f"Max difference : "
    f"{np.nanmax(seasonal_diff):.12f}"
)

print(
    f"Rows > 1e-10   : "
    f"{(seasonal_diff > 1e-10).sum():,}"
)

# ------------------------------------------------------------
# B3 metrics
# ------------------------------------------------------------

b3_aug_metrics = august_metrics(
    b3_aug_data["Target_Volume"],
    b3_aug_pred
)

# ------------------------------------------------------------
# Matched R0 and R1 on EXACT same B3 rows
# ------------------------------------------------------------

b3_matched_aug = b3_aug_data[
    [
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp",
        "Target_Volume"
    ]
].copy()

b3_matched_aug[
    "B3_SeasonalNaive_Prediction"
] = b3_aug_pred

b3_matched_aug = b3_matched_aug.merge(
    august_preds[
        [
            "Injection_Event_ID",
            "Detector_Code",
            "Timestamp",
            "R0_Injected_Prediction",
            "Final_R1_Injected_Prediction"
        ]
    ],
    on=[
        "Injection_Event_ID",
        "Detector_Code",
        "Timestamp"
    ],
    how="left",
    validate="one_to_one"
)

r0_b3_matched_metrics = august_metrics(
    b3_matched_aug["Target_Volume"],
    b3_matched_aug["R0_Injected_Prediction"]
)

r1_b3_matched_metrics = august_metrics(
    b3_matched_aug["Target_Volume"],
    b3_matched_aug["Final_R1_Injected_Prediction"]
)

# ------------------------------------------------------------
# Event-level availability
# ------------------------------------------------------------

b3_event_availability_aug = (
    august_baseline
    .assign(
        B3_Available=
        august_baseline[
            "Lag_96_Injected"
        ].notna()
    )
    .groupby("Injection_Event_ID")
    .agg(
        B3_Event_Fully_Available=(
            "B3_Available",
            "all"
        ),
        B3_Event_Any_Available=(
            "B3_Available",
            "any"
        )
    )
    .reset_index()
)

b3_full_events_aug = (
    b3_event_availability_aug[
        "B3_Event_Fully_Available"
    ].sum()
)

b3_any_events_aug = (
    b3_event_availability_aug[
        "B3_Event_Any_Available"
    ].sum()
)

total_aug_events = len(
    b3_event_availability_aug
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("B3 PERFORMANCE ON AVAILABLE AUGUST ROWS")
print("=" * 95)

print(
    f"B3 WAPE : "
    f"{b3_aug_metrics['WAPE']:.6f}%"
)

print(
    f"B3 MAE  : "
    f"{b3_aug_metrics['MAE']:.6f}"
)

print(
    f"B3 RMSE : "
    f"{b3_aug_metrics['RMSE']:.6f}"
)

print(
    f"B3 Bias : "
    f"{b3_aug_metrics['Bias']:.6f}"
)

print("\nMatched R0/R1 on SAME B3 rows:")

print(
    f"R0 matched WAPE : "
    f"{r0_b3_matched_metrics['WAPE']:.6f}%"
)

print(
    f"R1 matched WAPE : "
    f"{r1_b3_matched_metrics['WAPE']:.6f}%"
)

print(
    f"B3 minus R1     : "
    f"{b3_aug_metrics['WAPE'] - r1_b3_matched_metrics['WAPE']:+.4f} pp"
)

print("\nEvent-level availability:")

print(
    f"Fully available events : "
    f"{b3_full_events_aug:,}/{total_aug_events:,} "
    f"({100*b3_full_events_aug/total_aug_events:.2f}%)"
)

print(
    f"At least partly available events : "
    f"{b3_any_events_aug:,}/{total_aug_events:,} "
    f"({100*b3_any_events_aug/total_aug_events:.2f}%)"
)

# ------------------------------------------------------------
# Add B3 to saved practical-baseline file
# ------------------------------------------------------------

aug_practical_output[
    "B3_SeasonalNaive_Available"
] = (
    august_baseline[
        "Lag_96_Injected"
    ].notna()
)

aug_practical_output[
    "B3_SeasonalNaive_Prediction"
] = (
    august_baseline[
        "Lag_96_Injected"
    ]
)

aug_practical_output.to_parquet(
    aug_practical_output_path,
    index=False
)

print("\nUpdated saved file:")
print(aug_practical_output_path)

In [ ]:
# ============================================================
# CELL 40
# FINAL AUGUST PRACTICAL BASELINE TABLE
# + NOTEBOOK 6 CLOSURE
# ============================================================

import json
import pandas as pd
import numpy as np

print("=" * 110)
print("FINAL AUGUST MITIGATION AND PRACTICAL-BASELINE SUMMARY")
print("=" * 110)

# ------------------------------------------------------------
# Event availability for B2
# ------------------------------------------------------------

b2_full_events_aug = int(
    aug_event_availability[
        "B2_Event_Fully_Available"
    ].sum()
)

# ------------------------------------------------------------
# Final manuscript-ready table
# ------------------------------------------------------------

final_august_table = pd.DataFrame([
    {
        "Method": "R0 Clean-only",
        "Mitigation": "None",
        "Evaluation_Rows": len(august_baseline),
        "Row_Availability_Percent": 100.0,
        "Fully_Available_Events": 604,
        "Event_Availability_Percent": 100.0,
        "WAPE_Percent": r0_aug_metrics["WAPE"],
        "MAE": r0_aug_metrics["MAE"],
        "RMSE": r0_aug_metrics["RMSE"],
        "Bias": r0_aug_metrics["Bias"],
        "Evaluation_Basis": "Full August controlled set"
    },
    {
        "Method": "R1 Failure-aware",
        "Mitigation": "Failure-aware training",
        "Evaluation_Rows": len(august_baseline),
        "Row_Availability_Percent": 100.0,
        "Fully_Available_Events": 604,
        "Event_Availability_Percent": 100.0,
        "WAPE_Percent": r1_aug_metrics["WAPE"],
        "MAE": r1_aug_metrics["MAE"],
        "RMSE": r1_aug_metrics["RMSE"],
        "Bias": r1_aug_metrics["Bias"],
        "Evaluation_Basis": "Full August controlled set"
    },
    {
        "Method": "B1 Previous-day imputation",
        "Mitigation": "Replace missing lag using same lag position from previous day",
        "Evaluation_Rows": len(august_baseline),
        "Row_Availability_Percent": 100.0,
        "Fully_Available_Events": 604,
        "Event_Availability_Percent": 100.0,
        "WAPE_Percent": b1_aug_metrics["WAPE"],
        "MAE": b1_aug_metrics["MAE"],
        "RMSE": b1_aug_metrics["RMSE"],
        "Bias": b1_aug_metrics["Bias"],
        "Evaluation_Basis": "Full August controlled set"
    },
    {
        "Method": "B2 Previous-week imputation",
        "Mitigation": "Replace missing lag using same lag position from previous week",
        "Evaluation_Rows": len(b2_aug_data),
        "Row_Availability_Percent":
            100 * len(b2_aug_data) / len(august_baseline),
        "Fully_Available_Events": b2_full_events_aug,
        "Event_Availability_Percent":
            100 * b2_full_events_aug / 604,
        "WAPE_Percent": b2_aug_metrics["WAPE"],
        "MAE": b2_aug_metrics["MAE"],
        "RMSE": b2_aug_metrics["RMSE"],
        "Bias": b2_aug_metrics["Bias"],
        "Evaluation_Basis": "Available August rows only"
    },
    {
        "Method": "B3 Seasonal-naive fallback",
        "Mitigation": "Forecast from same interval 24 h earlier when available",
        "Evaluation_Rows": len(b3_aug_data),
        "Row_Availability_Percent":
            100 * len(b3_aug_data) / len(august_baseline),
        "Fully_Available_Events": int(b3_full_events_aug),
        "Event_Availability_Percent":
            100 * b3_full_events_aug / 604,
        "WAPE_Percent": b3_aug_metrics["WAPE"],
        "MAE": b3_aug_metrics["MAE"],
        "RMSE": b3_aug_metrics["RMSE"],
        "Bias": b3_aug_metrics["Bias"],
        "Evaluation_Basis": "Available August rows only"
    }
])

# ------------------------------------------------------------
# Improvements relative to R0
# Only directly report full-set improvement when coverage = 100%
# ------------------------------------------------------------

final_august_table[
    "WAPE_Improvement_vs_R0_Percent"
] = np.nan

full_coverage_mask = (
    final_august_table[
        "Row_Availability_Percent"
    ] == 100.0
)

final_august_table.loc[
    full_coverage_mask,
    "WAPE_Improvement_vs_R0_Percent"
] = (
    (
        r0_aug_metrics["WAPE"]
        - final_august_table.loc[
            full_coverage_mask,
            "WAPE_Percent"
        ]
    )
    / r0_aug_metrics["WAPE"]
    * 100
)

# ------------------------------------------------------------
# Display rounded table
# ------------------------------------------------------------

display_table = final_august_table.copy()

for col in [
    "Row_Availability_Percent",
    "Event_Availability_Percent",
    "WAPE_Percent",
    "MAE",
    "RMSE",
    "Bias",
    "WAPE_Improvement_vs_R0_Percent"
]:
    display_table[col] = display_table[col].round(2)

display(display_table)

# ------------------------------------------------------------
# Key final results
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("KEY FINAL HOLDOUT RESULTS")
print("=" * 110)

print(
    f"R0 Clean-only WAPE          : "
    f"{r0_aug_metrics['WAPE']:.2f}%"
)

print(
    f"R1 Failure-aware WAPE       : "
    f"{r1_aug_metrics['WAPE']:.2f}%"
)

print(
    f"B1 Previous-day WAPE        : "
    f"{b1_aug_metrics['WAPE']:.2f}%"
)

print(
    f"B1 availability             : 100.00%"
)

print(
    f"B1 improvement vs R0        : "
    f"{b1_vs_r0_improvement:.2f}%"
)

print(
    f"B1 difference vs R1         : "
    f"{b1_aug_metrics['WAPE'] - r1_aug_metrics['WAPE']:+.2f} pp"
)

print("\nPartial-coverage baselines:")

print(
    f"B2 Previous-week WAPE       : "
    f"{b2_aug_metrics['WAPE']:.2f}%"
)

print(
    f"B2 row availability         : "
    f"{100*len(b2_aug_data)/len(august_baseline):.2f}%"
)

print(
    f"B2 fully available events   : "
    f"{b2_full_events_aug}/604 "
    f"({100*b2_full_events_aug/604:.2f}%)"
)

print(
    f"B2 matched R1 WAPE          : "
    f"{r1_b2_matched_metrics['WAPE']:.2f}%"
)

print(
    f"B2 difference vs matched R1 : "
    f"{b2_aug_metrics['WAPE'] - r1_b2_matched_metrics['WAPE']:+.2f} pp"
)

print()

print(
    f"B3 Seasonal-naive WAPE      : "
    f"{b3_aug_metrics['WAPE']:.2f}%"
)

print(
    f"B3 row availability         : "
    f"{100*len(b3_aug_data)/len(august_baseline):.2f}%"
)

print(
    f"B3 fully available events   : "
    f"{b3_full_events_aug}/604 "
    f"({100*b3_full_events_aug/604:.2f}%)"
)

print(
    f"B3 matched R1 WAPE          : "
    f"{r1_b3_matched_metrics['WAPE']:.2f}%"
)

print(
    f"B3 difference vs matched R1 : "
    f"{b3_aug_metrics['WAPE'] - r1_b3_matched_metrics['WAPE']:+.2f} pp"
)

# ------------------------------------------------------------
# Save final table
# ------------------------------------------------------------

final_august_table_path = (
    NB6_TABLES /
    "Table_FINAL_August_Practical_Baseline_Comparison.csv"
)

final_august_table.to_csv(
    final_august_table_path,
    index=False
)

# ------------------------------------------------------------
# Save concise final results JSON
# ------------------------------------------------------------

final_results = {
    "August_Controlled_Events": 604,
    "August_Affected_Rows": 12662,

    "R0_WAPE": float(r0_aug_metrics["WAPE"]),
    "R1_WAPE": float(r1_aug_metrics["WAPE"]),

    "B1_WAPE": float(b1_aug_metrics["WAPE"]),
    "B1_MAE": float(b1_aug_metrics["MAE"]),
    "B1_RMSE": float(b1_aug_metrics["RMSE"]),
    "B1_Bias": float(b1_aug_metrics["Bias"]),
    "B1_Row_Availability_Percent": 100.0,
    "B1_Event_Availability_Percent": 100.0,
    "B1_Improvement_vs_R0_Percent":
        float(b1_vs_r0_improvement),
    "B1_minus_R1_WAPE_pp":
        float(
            b1_aug_metrics["WAPE"]
            - r1_aug_metrics["WAPE"]
        ),

    "B2_WAPE_AvailableRows":
        float(b2_aug_metrics["WAPE"]),
    "B2_Row_Availability_Percent":
        float(
            100 * len(b2_aug_data)
            / len(august_baseline)
        ),
    "B2_Event_Availability_Percent":
        float(
            100 * b2_full_events_aug / 604
        ),
    "B2_Matched_R1_WAPE":
        float(r1_b2_matched_metrics["WAPE"]),

    "B3_WAPE_AvailableRows":
        float(b3_aug_metrics["WAPE"]),
    "B3_Row_Availability_Percent":
        float(
            100 * len(b3_aug_data)
            / len(august_baseline)
        ),
    "B3_Event_Availability_Percent":
        float(
            100 * b3_full_events_aug / 604
        ),
    "B3_Matched_R1_WAPE":
        float(r1_b3_matched_metrics["WAPE"])
}

final_json_path = (
    NB6_RESULTS /
    "FINAL_August_Practical_Baseline_Results.json"
)

with open(
    final_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_results,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

required_final_files = [
    aug_practical_output_path,
    final_august_table_path,
    final_json_path
]

print("\n" + "=" * 110)
print("FINAL NOTEBOOK 6 OUTPUT AUDIT")
print("=" * 110)

all_final_found = True

for path in required_final_files:

    exists = path.exists()

    if not exists:
        all_final_found = False

    print(
        f"{'FOUND' if exists else 'MISSING':<8} | "
        f"{path.name}"
    )

print("\n" + "=" * 110)

if all_final_found:
    print(
        "PASS — NOTEBOOK 6 FINAL AUGUST ANALYSIS COMPLETE."
    )
else:
    print(
        "CHECK REQUIRED — One or more final files are missing."
    )

print("=" * 110)

print("\nFinal manuscript table:")
print(final_august_table_path)

print("\nFinal results JSON:")
print(final_json_path)